In [ ]:
import asyncio
import os
from google import genai
from google.genai import types
# 1. Define a tool the model can call
def get_current_time():
    """Returns the current system time."""
    from datetime import datetime
    return {"time": datetime.now().strftime("%H:%M:%S")}

# 2. Setup Client
# Ensure you use 'v1alpha' for preview features
client = genai.Client(
    api_key="YOUR_API_KEY", 
    http_options={'api_version': 'v1alpha'}
)

MODEL_ID = "gemini-2.5-flash-native-audio-preview-12-2025"
# types.LiveServerContent()
async def main():
    # 3. Configure the session for audio + text + tools
    config = types.LiveConnectConfig(
        tools=[get_current_time],
        response_modalities=["AUDIO"], # Returns native audio
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        # This returns the text transcript of the model's audio response
        output_audio_transcription=types.AudioTranscriptionConfig()
    )

    async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
        # await session.send_tool_response()
        # Send text input or audio bytes (e.g., from a mic stream)
        # To send audio, replace the string with audio bytes:
        # await session.send(audio_bytes, end_of_turn=True)
        await session.send_tool_response(function_responses=[types.FunctionResponse(name="get_current_time", id="123", response="12:34:56")])
        await session.send_realtime_input()
        await session.send_client_content(turns= types.Content(role="user", parts=[types.Part(text="What time is it?")]), turn_complete=True ,)

        async for message in session.receive():
            # A. HANDLE FUNCTION CALLS
            # message.tool_call.function_calls
            # message.server_content.model_turn.parts
            # message.server_content.output_transcription.


            if message.tool_call:
                if message.tool_call.function_calls:
                    for call in message.tool_call.function_calls:
                        print(f"Executing: {call.name}")
                        await session.send(
                            types.LiveClientToolResponse(
                                function_responses=[
                                    types.FunctionResponse(name=call.name, id=call.id, response=result)
                                ]
                            )
                        )

            # B. HANDLE TEXT & AUDIO OUTPUT
            if message.server_content and message.server_content.model_turn:
                for part in message.server_content.model_turn.parts:
                    # Capture Text Transcript
                    if part.text:
                        print(f"Model Transcript: {part.text}")
                    
                    # Capture Audio Data
                    if part.inline_data:
                        # part.inline_data.data contains the base64 audio bytes
                        print("Received audio chunk...")

if __name__ == "__main__":
    asyncio.run(main())


In [1]:
response="setup_complete=None server_content=LiveServerContent(\n  input_transcription=Transcription(\n    text=' Hi, can you help me?'\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)

setup_complete=None server_content=LiveServerContent(
  input_transcription=Transcription(
    text=' Hi, can you help me?'
  )
) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None


In [2]:
esponse='setup_complete=None server_content=LiveServerContent(\n  model_turn=Content(\n    parts=[\n      Part(\n        text="""**Acknowledge Shopping Request**\n\nI\'ve registered the user\'s initial greeting and request for assistance. My focus is now on responding in a friendly way. I\'ll frame my response as an offer to help with their shopping, keeping it concise and helpful. I need to be sure to adhere to the "friendly Voice Shopping Assistant" instructions.\n\n\n""",\n        thought=True\n      ),\n    ]\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None'
print (esponse)

setup_complete=None server_content=LiveServerContent(
  model_turn=Content(
    parts=[
      Part(
        text="""**Acknowledge Shopping Request**

I've registered the user's initial greeting and request for assistance. My focus is now on responding in a friendly way. I'll frame my response as an offer to help with their shopping, keeping it concise and helpful. I need to be sure to adhere to the "friendly Voice Shopping Assistant" instructions.


""",
        thought=True
      ),
    ]
  )
) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None


In [3]:
response="setup_complete=None server_content=LiveServerContent(\n  turn_complete=True\n) tool_call=None tool_call_cancellation=None usage_metadata=UsageMetadata(\n  prompt_token_count=1021,\n  prompt_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.TEXT: 'TEXT'>,\n      token_count=1020\n    ),\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=1\n    ),\n  ],\n  response_token_count=63,\n  response_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=63\n    ),\n  ],\n  thoughts_token_count=47,\n  total_token_count=1084\n) go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)

setup_complete=None server_content=LiveServerContent(
  turn_complete=True
) tool_call=None tool_call_cancellation=None usage_metadata=UsageMetadata(
  prompt_token_count=1021,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1020
    ),
    ModalityTokenCount(
      modality=<MediaModality.AUDIO: 'AUDIO'>,
      token_count=1
    ),
  ],
  response_token_count=63,
  response_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.AUDIO: 'AUDIO'>,
      token_count=63
    ),
  ],
  thoughts_token_count=47,
  total_token_count=1084
) go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None


In [4]:
response="setup_complete=None server_content=LiveServerContent(\n  model_turn=Content(\n    parts=[\n      Part(\n        inline_data=Blob(\n          data=b'\\xf4\\xff\\xf3\\xff\\xf6\\xff\\xf5\\xff\\xf8\\xff\\xf7\\xff\\xfb\\xff\\xfb\\xff\\xfd\\xff\\xfe\\xff\\xff\\xff\\xfd\\xff\\x00\\x00\\x02\\x00\\x03\\x00\\x03\\x00\\x03\\x00\\x04\\x00\\x04\\x00\\x05\\x00\\x04\\x00\\x04\\x00\\x03\\x00\\x05\\x00\\x04\\x00\\x08\\x00\\x07\\x00\\n\\x00\\t\\x00\\x0e\\x00\\x0b\\x00\\x0b\\x00\\x0c\\x00\\x0c\\x00\\x0b\\x00\\n\\x00\\t\\x00\\n\\x00\\x08\\x00\\x08\\x00\\x06\\x00\\x05\\x00\\x03\\x00\\x03\\x00\\x02\\x00\\xff\\xff\\xff\\xff\\xfd\\xff\\xfa...',\n          mime_type='audio/pcm;rate=24000'\n        )\n      ),\n    ]\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)

setup_complete=None server_content=LiveServerContent(
  model_turn=Content(
    parts=[
      Part(
        inline_data=Blob(
          data=b'\xf4\xff\xf3\xff\xf6\xff\xf5\xff\xf8\xff\xf7\xff\xfb\xff\xfb\xff\xfd\xff\xfe\xff\xff\xff\xfd\xff\x00\x00\x02\x00\x03\x00\x03\x00\x03\x00\x04\x00\x04\x00\x05\x00\x04\x00\x04\x00\x03\x00\x05\x00\x04\x00\x08\x00\x07\x00\n\x00\t\x00\x0e\x00\x0b\x00\x0b\x00\x0c\x00\x0c\x00\x0b\x00\n\x00\t\x00\n\x00\x08\x00\x08\x00\x06\x00\x05\x00\x03\x00\x03\x00\x02\x00\xff\xff\xff\xff\xfd\xff\xfa...',
          mime_type='audio/pcm;rate=24000'
        )
      ),
    ]
  )
) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None


In [1]:
response="setup_complete=None server_content=LiveServerContent(\n  turn_complete=True\n) tool_call=None tool_call_cancellation=None usage_metadata=UsageMetadata(\n  prompt_token_count=3732,\n  prompt_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.TEXT: 'TEXT'>,\n      token_count=3624\n    ),\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=108\n    ),\n  ],\n  response_token_count=103,\n  response_tokens_details=[\n    ModalityTokenCount(\n      modality=<MediaModality.AUDIO: 'AUDIO'>,\n      token_count=103\n    ),\n  ],\n  thoughts_token_count=66,\n  total_token_count=3835\n) go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None"
print (response)


setup_complete=None server_content=LiveServerContent(
  turn_complete=True
) tool_call=None tool_call_cancellation=None usage_metadata=UsageMetadata(
  prompt_token_count=3732,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3624
    ),
    ModalityTokenCount(
      modality=<MediaModality.AUDIO: 'AUDIO'>,
      token_count=108
    ),
  ],
  response_token_count=103,
  response_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.AUDIO: 'AUDIO'>,
      token_count=103
    ),
  ],
  thoughts_token_count=66,
  total_token_count=3835
) go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None


In [9]:
response='setup_complete=None server_content=LiveServerContent(\n  model_turn=Content(\n    parts=[\n      Part(\n        text="""**Confirming Specific Product**\n\nI\'ve determined the user wants the "Nike Air Zoom Pegasus 40" with product ID 1, but I need to ensure this is the intended product. The user\'s use of "one" confirms a quantity of one, so I will now verify this specific item is correct based on their previous searches. I must verify this selection before proceeding.\n\n\n""",\n        thought=True\n      ),\n    ]\n  )\n) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None'
print (response)

setup_complete=None server_content=LiveServerContent(
  model_turn=Content(
    parts=[
      Part(
        text="""**Confirming Specific Product**

I've determined the user wants the "Nike Air Zoom Pegasus 40" with product ID 1, but I need to ensure this is the intended product. The user's use of "one" confirms a quantity of one, so I will now verify this specific item is correct based on their previous searches. I must verify this selection before proceeding.


""",
        thought=True
      ),
    ]
  )
) tool_call=None tool_call_cancellation=None usage_metadata=None go_away=None session_resumption_update=None voice_activity_detection_signal=None voice_activity=None


In [1]:
memory=[{'role': 'user', 'text': 'Hi'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' How'}, {'role': 'user', 'text': ' are'}, {'role': 'user', 'text': ' you'}, {'role': 'user', 'text': '?'}, {'role': 'ai', 'text': "I'm doing well, thank you! Are you looking for anything specific in the store?"}, {'role': 'user', 'text': ' can'}, {'role': 'user', 'text': ' you'}, {'role': 'user', 'text': ' help'}, {'role': 'user', 'text': ' me'}, {'role': 'user', 'text': ' with showing?'}, {'role': 'ai', 'text': 'Sure! What kind of products are you interested in?'}, {'role': 'user', 'text': 'Lest show what available'}, {'role': 'ai', 'text': 'What kind of products are you looking for? For example, shoes, electronics, or clothing?'}]
memory=[{'role': 'user', 'text': 'Hi'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': 'What is your name'}, {'role': 'ai', 'text': "I don't have a specific name, I'm your voice shopping assistant."}, {'role': 'ai', 'text': 'I can search for products, get details about them, show your cart, and add or remove items. What are you looking for?'}, {'role': 'user', 'text': ' Okay, what you can help me with?'}, {'role': 'ai', 'text': 'How about some running shoes? Or are you interested in something else?'}, {'role': 'user', 'text': ' Uh anything'}]
memory=[{'role': 'user', 'text': 'Hi how are you'}, {'role': 'ai', 'text': "Hello! I'm good, thank you. How can I help you with your shopping today?"}, {'role': 'ai', 'text': 'I am your voice shopping assistant. How can I help you find anything?'}, {'role': 'user', 'text': " What's your name?"}, {'role': 'ai', 'text': 'I can help you search for products, check your shopping cart, or provide product details. What are you looking for today?'}, {'role': 'user', 'text': ' what you can meet me and brothers'}, {'role': 'ai', 'text': 'We currently have the North Face ThermoBall Jacket available.'}, {'role': 'user', 'text': ' Вот какие же продукты с'}, {'role': 'user', 'text': 'What others'}, {'role': 'ai', 'text': "Is there a specific category or type of product you're interested in?"}]
memory=[{'role': 'user', 'text': 'Hi how aer o'}, {'role': 'ai', 'text': 'I am good, thank you! How can I help you with your shopping today?'}, {'role': 'user', 'text': 'was your IM'}, {'role': 'ai', 'text': "I don't have a name. I'm just a voice shopping assistant. What are you looking for?"}, {'role': 'user', 'text': 'what is product zero'}, {'role': 'ai', 'text': "I'm sorry, but I can't find a product with product ID 0. Is there anything specific you're searching for?"}, {'role': 'user', 'text': 'I mean what is category'}, {'role': 'ai', 'text': 'There\'s no category called "zero". What type of products are you interested in?'}, {'role': 'user', 'text': 'Okay I mean what what is available'}, {'role': 'ai', 'text': 'We have various categories like shoes, clothing, and electronics. What would you like to browse?'}, {'role': 'user', 'text': 'Great lest start with electronics'}, {'role': 'ai', 'text': 'I\'m sorry, but I didn\'t find any products in the "electronics" category. Is there another category you\'re interested in?'}]
memory=[{'role': 'user', 'text': ' Hi, how are you?'}, {'role': 'ai', 'text': "Hello! I'm good, thank you. How can I help you with your shopping today?"}, {'role': 'user', 'text': ' what you can do with'}, {'role': 'ai', 'text': 'I can search for products, show you details, manage your cart, and more. What are you looking for?'}, {'role': 'user', 'text': ' you want to hit it.'}, {'role': 'ai', 'text': "Sorry, I didn't find any products matching 'headset'. Is there something else you're interested in?"}, {'role': 'user', 'text': ' what about?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99 and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': ' what you'}, {'role': 'ai', 'text': 'Both are great choices! The Sony is renowned for noise cancellation and long battery life, while the Apple provides excellent active noise cancellation and transparency modes. It really depends on your priority.'}, {'role': 'user', 'text': ' I think we can go with Sony.'}, {'role': 'ai', 'text': 'Sure! Shall I add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': 'Add 2 of it'}, {'role': 'user', 'text': ' yes added.'}, {'role': 'ai', 'text': 'Done! I have added the Sony WH-1000XM5 to your cart. Your current total is $349.99. Anything else you need?'}, {'role': 'ai', 'text': 'You currently have one in your cart. Shall I change the quantity to 2?'}, {'role': 'user', 'text': 'Yes and add on air pods'}, {'role': 'ai', 'text': "I've updated the Sony quantity to 2 and added the Apple AirPods Pro. Your total is $1299.96. Is there anything else?"}, {'role': 'user', 'text': 'This is good form you,. So on ht cart'}, {'role': 'ai', 'text': 'Your cart has 3x Sony WH-1000XM5 and 1x Apple AirPods Pro, for a total of $1299.96. Are you ready to checkout?'}, {'role': 'user', 'text': 'Add some hoodies'}, {'role': 'ai', 'text': "I'm sorry, I couldn't find any hoodies. Is there something else you were looking for?"}, {'role': 'user', 'text': ' you can search for'}, {'role': 'ai', 'text': 'I found two wireless earbuds: the Sony WH-1000XM5 for $349.99 and the Apple AirPods Pro (2nd Gen) for $249.99. Did you want to add one of those?'}, {'role': 'user', 'text': ' You can search for holdings.'}, {'role': 'ai', 'text': "As I mentioned before, I can't find any hoodies. Perhaps a jacket or sweater?"}, {'role': 'user', 'text': ' try to get all this'}, {'role': 'ai', 'text': 'Could you clarify what "all this" you would like to get? Are you referring to the Sony and Apple earbuds in your cart?'}, {'role': 'user', 'text': ' to all products of clauses.'}, {'role': 'ai', 'text': 'I\'m sorry, I can only search for products by keywords or specific categories, and I don\'t have a general "clothes" category. What specific types of clothing are you interested in?'}, {'role': 'user', 'text': " I'm sure this in the product is as category was closing."}, {'role': 'ai', 'text': "I understand, but my search function didn't find any hoodies using the 'clothing' category. We can try searching for specific types of clothing or other terms if you'd like."}, {'role': 'user', 'text': 'Are you sure you cna find this product Champion Reverse Weave Hoodie'}, {'role': 'ai', 'text': 'Yes, I found it! We have the Champion Reverse Weave Hoodie for $59.99. Would you like to add it to your cart?'}, {'role': 'user', 'text': " Yes, I didn't."}, {'role': 'ai', 'text': "Great! I've added the Champion Hoodie to your cart. Is there anything else?"}, {'role': 'user', 'text': ' no'}, {'role': 'ai', 'text': "Okay. Your current total is $1359.95. Let me know if you're ready to checkout."}, {'role': 'user', 'text': ' Yes. Yes.'}, {'role': 'ai', 'text': 'Great! To complete your purchase, please proceed to the checkout area. Is there anything else I can assist you with today?'}, {'role': 'user', 'text': ' What is final chapter'}, {'role': 'ai', 'text': 'Your final total is $1359.95.'}, {'role': 'user', 'text': ' . Give me fall card.'}, {'role': 'ai', 'text': 'Certainly. Your cart contains: 2x Sony WH-1000XM5, 1x Apple AirPods Pro (2nd Gen), and 1x Champion Reverse Weave Hoodie, for a total of $1359.95. Is that correct?'}, {'role': 'user', 'text': ' yes'}, {'role': 'ai', 'text': 'Perfect! You can proceed to checkout to finalize your order. Thank you for shopping with us!'}]

for msg in memory:
    print(msg)


{'role': 'user', 'text': ' Hi, how are you?'}
{'role': 'ai', 'text': "Hello! I'm good, thank you. How can I help you with your shopping today?"}
{'role': 'user', 'text': ' what you can do with'}
{'role': 'ai', 'text': 'I can search for products, show you details, manage your cart, and more. What are you looking for?'}
{'role': 'user', 'text': ' you want to hit it.'}
{'role': 'ai', 'text': "Sorry, I didn't find any products matching 'headset'. Is there something else you're interested in?"}
{'role': 'user', 'text': ' what about?'}
{'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99 and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}
{'role': 'user', 'text': ' what you'}
{'role': 'ai', 'text': 'Both are great choices! The Sony is renowned for noise cancellation and long battery life, while the Apple provides excellent active noise cancellation and transparency modes. It really depends on your priority.'}
{'role': 'user', 'text': ' I think

In [1]:
memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}, {'role': 'user', 'text': 'Ues and add 2 of it'}, {'role': 'ai', 'text': "I've added 2 quantities of the Levi's 501 Jeans to your cart. Is there anything else I can help you with?"}, {'role': 'user', 'text': ' Wat een kracht.'}, {'role': 'ai', 'text': "Your cart contains: 1 Champion Reverse Weave Hoodie, 1 Sony WH-1000XM5 and 2 Levi's 501 Original Jeans. The total is $549.96. Ready to checkout?"}]
for item in memory:
    print(item)
   

{'role': 'user', 'text': ' Thôi.'}
{'role': 'ai', 'text': 'Hello! How can I help you today?'}
{'role': 'user', 'text': ' What is my name?'}
{'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}
{'role': 'user', 'text': ' Okay my name is'}
{'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}
{'role': 'user', 'text': ' اي والله دي'}
{'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}
{'role': 'user', 'text': ' I mean I want a holy closing.'}
{'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}
{'role': 'user', 'text': ' Okay second one.'}
{'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}


2026-03-26T22:47:23.556504Z
2026-03-26T22:49:59.371183Z 


026-03-26T22:53:11.917993Z

I've determined the user wants the "Nike Air Zoom Pegasus 40" with product ID 1, but I need to ensure this is the intended product. The user's use of "one" confirms a quantity of one, so I will now verify this specific item is correct based on their previous searches. I must verify this selection before proceeding

(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent>
 *  History restored 

PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> 
 *  History restored 

PS P:\AI_Empire\Projects\voice-commerce-agent> & p:\AI_Empire\Projects\voice-commerce-agent\.venv\Scripts\Activate.ps1
(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> uv run uvicorn src.voice_commerce.main:app --reload
INFO:     Will watch for changes in these directories: ['P:\\AI_Empire\\Projects\\voice-commerce-agent']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [19424] using WatchFiles
INFO:     Started server process [19312]
INFO:     Waiting for application startup.
2026-03-26T20:12:13.053031Z [info     ] voice_commerce_agent_starting  [src.voice_commerce.main] debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-26T20:12:16.679647Z [info     ] voice_handler_created          [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
INFO:     127.0.0.1:10612 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-26T20:12:16.680167Z [info     ] ws_connected                   [voice_commerce.handlers.voice_websocket_handler] client="Address(host='127.0.0.1', port=10612)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-26T20:12:16.941470Z [info     ] gemini_session_connecting      [voice_commerce.core.voice.gemini_live_handler] model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-26T20:12:17.608773Z [info     ] gemini_session_connected       [voice_commerce.core.voice.gemini_live_handler]
INFO:     127.0.0.1:6023 - "GET /static/test_client.html HTTP/1.1" 304 Not Modified
INFO:     connection closed
2026-03-26T20:12:18.376701Z [info     ] gemini_session_closing         [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:12:18.534793Z [info     ] gemini_session_closed_by_server [voice_commerce.core.voice.gemini_live_handler] reason='1000 None. '
2026-03-26T20:12:18.535323Z [info     ] gemini_session_closed_gracefully [voice_commerce.handlers.voice_websocket_handler] reason='1000 None. ' session=v5_mph3j0z2
2026-03-26T20:12:18.535847Z [info     ] gemini_session_closed          [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:12:18.536377Z [info     ] ws_disconnected_normally       [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 turns=0
2026-03-26T20:12:18.536377Z [info     ] ws_handler_finished            [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 total_turns=0
2026-03-26T20:12:18.812149Z [info     ] voice_handler_created          [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
INFO:     127.0.0.1:5957 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-26T20:12:18.812149Z [info     ] ws_connected                   [voice_commerce.handlers.voice_websocket_handler] client="Address(host='127.0.0.1', port=5957)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-26T20:12:18.888479Z [info     ] gemini_session_connecting      [voice_commerce.core.voice.gemini_live_handler] model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-26T20:12:20.015171Z [info     ] gemini_session_connected       [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:12:20.711320Z [info     ] input_mode_switched_to_audio   [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
2026-03-26T20:12:24.232599Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Thôi.'
INFO:     127.0.0.1:12462 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-03-26T20:12:26.866114Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}]
2026-03-26T20:12:30.713359Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' What'
2026-03-26T20:12:30.714893Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' is my name?'
2026-03-26T20:12:38.088775Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}]
2026-03-26T20:12:43.111597Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Okay'
2026-03-26T20:12:44.907326Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' my'
2026-03-26T20:12:44.909488Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' name'
2026-03-26T20:12:44.910510Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' is'
2026-03-26T20:12:49.085829Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}]
2026-03-26T20:13:00.345693Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' اي والله دي'
2026-03-26T20:13:00.706061Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'هدايا'} call_id=function-call-12340314771335619223 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:00.707093Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'هدايا'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:13:00.707624Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'هدايا'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:00.708661Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=هدايا
2026-03-26T20:13:00.710162Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=100 tool=search_products
2026-03-26T20:13:00.711746Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="I didn't find anything matching 'هدايا'. Try different keywords, or ask me what " session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:00.713549Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'هدايا'} call_id=function-call-12340314771335619223 function_call=FunctionCall(      
  args={
    'query': 'هدايا'
  },
  id='function-call-12340314771335619223',
  name='search_products'
)
2026-03-26T20:13:11.100242Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}]
2026-03-26T20:13:14.686226Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' I'
2026-03-26T20:13:14.856672Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' mean'
2026-03-26T20:13:15.000955Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' I'
2026-03-26T20:13:15.368847Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' want'
2026-03-26T20:13:15.752175Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' a holy closing.'
2026-03-26T20:13:16.206677Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'hoodie clothing'} call_id=function-call-8040951186187757545 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:16.207227Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'hoodie clothing'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:13:16.207227Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'hoodie clothing'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:16.208281Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query='hoodie clothing'
2026-03-26T20:13:16.208788Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=296 tool=search_products
2026-03-26T20:13:16.209319Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Found 2 products for 'hoodie clothing':\n• Levi's 501 Original Jeans — $69.99 — i" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:16.210334Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'hoodie clothing'} call_id=function-call-8040951186187757545 function_call=FunctionCall(
  args={
    'query': 'hoodie clothing'
  },
  id='function-call-8040951186187757545',
  name='search_products'
)
2026-03-26T20:13:28.504365Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}]
2026-03-26T20:13:35.163588Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Okay'
2026-03-26T20:13:35.325770Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' second one.'
2026-03-26T20:13:39.582613Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}]
2026-03-26T20:13:42.903594Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' E ele'
2026-03-26T20:13:43.139460Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'product_id': 8, 'quantity': 1} call_id=function-call-7697120716237827354 session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:13:43.141072Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'product_id': 8, 'quantity': 1} session=v5_mph3j0z2 tool_name=add_to_cart
2026-03-26T20:13:43.141072Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'product_id': '8', 'quantity': '1'} session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:13:43.141072Z [info     ] add_to_cart                    [voice_commerce.core.tools.cart_tools] product_id=8 quantity=1 session=v5_mph3j0z2
2026-03-26T20:13:43.142598Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=94 tool=add_to_cart
2026-03-26T20:13:43.142598Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Added 1× Champion Reverse Weave Hoodie ($59.99 each) to your cart.\nCart total: $' session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:13:43.144180Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'product_id': 8, 'quantity': 1} call_id=function-call-7697120716237827354 function_call=FunctionCall(
  args={
    'product_id': 8,
    'quantity': 1
  },
  id='function-call-7697120716237827354',
  name='add_to_cart'
)
2026-03-26T20:13:49.637101Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}]  
2026-03-26T20:13:56.400325Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' wha'
2026-03-26T20:13:56.586759Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text='t is'
2026-03-26T20:13:56.616779Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' a very'
2026-03-26T20:13:56.864384Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'belt'} call_id=function-call-15346215065980089208 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:56.864902Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'belt'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:13:56.864902Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'belt'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:56.864902Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=belt
2026-03-26T20:13:56.866506Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=99 tool=search_products
2026-03-26T20:13:56.866506Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="I didn't find anything matching 'belt'. Try different keywords, or ask me what c" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:13:56.866506Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'belt'} call_id=function-call-15346215065980089208 function_call=FunctionCall(       
  args={
    'query': 'belt'
  },
  id='function-call-15346215065980089208',
  name='search_products'
)
2026-03-26T20:14:05.856963Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}]
2026-03-26T20:14:22.847509Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' ele'
2026-03-26T20:14:23.039275Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=ctro
2026-03-26T20:14:23.228339Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=nics
2026-03-26T20:14:23.289778Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text='. Can you hear me?'
2026-03-26T20:14:29.000453Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}]
2026-03-26T20:14:31.971384Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Yes.'
2026-03-26T20:14:32.320323Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'electronics', 'category': 'electronics'} call_id=function-call-2888346715497691110
 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:32.320323Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'electronics', 'category': 'electronics'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:14:32.321933Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'electronics', 'category': 'electronics'} session=v5_mph3j0z2 tool=search_products       
2026-03-26T20:14:32.322469Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=electronics
2026-03-26T20:14:32.322469Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=106 tool=search_products
2026-03-26T20:14:32.323034Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="I didn't find anything matching 'electronics'. Try different keywords, or ask me" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:32.323034Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'electronics', 'category': 'electronics'} call_id=function-call-2888346715497691110 function_call=FunctionCall(
  args={
    'category': 'electronics',
    'query': 'electronics'
  },
  id='function-call-2888346715497691110',
  name='search_products'
)
2026-03-26T20:14:44.154786Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}]
2026-03-26T20:14:47.151939Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Ye'
2026-03-26T20:14:47.345071Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=s
2026-03-26T20:14:47.537790Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' wha'
2026-03-26T20:14:47.538302Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text='t is headphone?'
2026-03-26T20:14:47.846869Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'headphones'} call_id=function-call-374181651143909031 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:47.846869Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'headphones'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:14:47.847920Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'headphones'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:47.848435Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=headphones
2026-03-26T20:14:47.848435Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=306 tool=search_products
2026-03-26T20:14:47.848435Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Found 2 products for 'headphones':\n• Sony WH-1000XM5 — $349.99 — in stock — ID:4" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:14:47.849956Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'headphones'} call_id=function-call-374181651143909031 function_call=FunctionCall(   
  args={
    'query': 'headphones'
  },
  id='function-call-374181651143909031',
  name='search_products'
)
2026-03-26T20:15:01.260870Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}]
2026-03-26T20:15:05.382782Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Okay'
2026-03-26T20:15:05.718132Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=', yeah'
2026-03-26T20:15:05.854176Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=,
2026-03-26T20:15:06.053346Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' let'
2026-03-26T20:15:06.185436Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text="'s"
2026-03-26T20:15:06.359878Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' go'
2026-03-26T20:15:06.530489Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' to'
2026-03-26T20:15:09.424562Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Sony.'
2026-03-26T20:15:15.093783Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}]
2026-03-26T20:15:24.004234Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Yes.'
2026-03-26T20:15:24.662894Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'product_id': 4, 'quantity': 1} call_id=function-call-14236623242608552161 session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:15:24.664421Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'product_id': 4, 'quantity': 1} session=v5_mph3j0z2 tool_name=add_to_cart
2026-03-26T20:15:24.664421Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'product_id': '4', 'quantity': '1'} session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:15:24.664421Z [info     ] add_to_cart                    [voice_commerce.core.tools.cart_tools] product_id=4 quantity=1 session=v5_mph3j0z2
2026-03-26T20:15:24.664421Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=83 tool=add_to_cart
2026-03-26T20:15:24.666071Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Added 1× Sony WH-1000XM5 ($349.99 each) to your cart.\nCart total: $409.98 (2 ite' session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:15:24.666602Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'product_id': 4, 'quantity': 1} call_id=function-call-14236623242608552161 function_call=FunctionCall(
  args={
    'product_id': 4,
    'quantity': 1
  },
  id='function-call-14236623242608552161',
  name='add_to_cart'
)
2026-03-26T20:15:30.056943Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}]
2026-03-26T20:15:34.531292Z [info     ] text_message_received          [voice_commerce.handlers.voice_websocket_handler] preview='What about jeans' session_id=v5_mph3j0z2
2026-03-26T20:15:34.935008Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'query': 'jeans'} call_id=function-call-8626671772119431720 session=v5_mph3j0z2 tool=search_products
2026-03-26T20:15:34.935562Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'query': 'jeans'} session=v5_mph3j0z2 tool_name=search_products
2026-03-26T20:15:34.935562Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'query': 'jeans'} session=v5_mph3j0z2 tool=search_products
2026-03-26T20:15:34.936576Z [info     ] search_products                [voice_commerce.core.tools.product_tools] max_price=None query=jeans
2026-03-26T20:15:34.936576Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=174 tool=search_products
2026-03-26T20:15:34.937126Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Found 1 product for 'jeans':\n• Levi's 501 Original Jeans — $69.99 — in stock — I" session=v5_mph3j0z2 tool=search_products
2026-03-26T20:15:34.938186Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'query': 'jeans'} call_id=function-call-8626671772119431720 function_call=FunctionCall(       
  args={
    'query': 'jeans'
  },
  id='function-call-8626671772119431720',
  name='search_products'
)
2026-03-26T20:15:42.316088Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}]
2026-03-26T20:15:48.066540Z [info     ] text_message_received          [voice_commerce.handlers.voice_websocket_handler] preview='Good add it' session_id=v5_mph3j0z2
2026-03-26T20:15:52.886734Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}]
2026-03-26T20:16:00.019591Z [info     ] text_message_received          [voice_commerce.handlers.voice_websocket_handler] preview='Ues and add 2 of it' session_id=v5_mph3j0z2
2026-03-26T20:16:00.497847Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={'product_id': 7, 'quantity': 2} call_id=function-call-6455127725345814488 session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:16:00.498377Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={'product_id': 7, 'quantity': 2} session=v5_mph3j0z2 tool_name=add_to_cart
2026-03-26T20:16:00.498377Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={'product_id': '7', 'quantity': '2'} session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:16:00.499391Z [info     ] add_to_cart                    [voice_commerce.core.tools.cart_tools] product_id=7 quantity=2 session=v5_mph3j0z2
2026-03-26T20:16:00.499898Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=92 tool=add_to_cart
2026-03-26T20:16:00.499898Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview="Added 2× Levi's 501 Original Jeans ($69.99 each) to your cart.\nCart total: $549." session=v5_mph3j0z2 tool=add_to_cart
2026-03-26T20:16:00.500948Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={'product_id': 7, 'quantity': 2} call_id=function-call-6455127725345814488 function_call=FunctionCall(
  args={
    'product_id': 7,
    'quantity': 2
  },
  id='function-call-6455127725345814488',
  name='add_to_cart'
)
2026-03-26T20:16:08.143949Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}, {'role': 'user', 'text': 'Ues and add 2 of it'}, {'role': 'ai', 'text': "I've added 2 quantities of the Levi's 501 Jeans to your cart. Is there anything else I can help you with?"}]
2026-03-26T20:16:08.519179Z [info     ] input_mode_switched_to_audio   [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
2026-03-26T20:16:11.301296Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' Wat een kracht.'
2026-03-26T20:16:11.598628Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={} call_id=function-call-3075882724188559665 session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:16:11.599643Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={} session=v5_mph3j0z2 tool_name=show_cart
2026-03-26T20:16:11.600158Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={} session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:16:11.600158Z [info     ] show_cart                      [voice_commerce.core.tools.cart_tools] session=v5_mph3j0z2
2026-03-26T20:16:11.600158Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=237 tool=show_cart
2026-03-26T20:16:11.601172Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Your cart:\n  • 1× Champion Reverse Weave Hoodie — $59.99 each = $59.99\n  • 1× So' session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:16:11.601172Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={} call_id=function-call-3075882724188559665 function_call=FunctionCall(
  args={},
  id='function-call-3075882724188559665',
  name='show_cart'
)
2026-03-26T20:16:27.545624Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'user', 'text': ' Thôi.'}, {'role': 'ai', 'text': 'Hello! How can I help you today?'}, {'role': 'user', 'text': ' What is my name?'}, {'role': 'ai', 'text': "I don't have personal information about you, like your name. Is there something specific I can assist you with?"}, {'role': 'user', 'text': ' Okay my name is'}, {'role': 'ai', 'text': "It's nice to meet you, Phoenix! How can I help you find in the store today?"}, {'role': 'user', 'text': ' اي والله دي'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'هدايا'. Perhaps you'd like to search for something specific, like 'perfumes' or 'accessories'?"}, {'role': 'user', 'text': ' I mean I want a holy closing.'}, {'role': 'ai', 'text': "I found two products for you: the Levi's 501 Original Jeans for $69.99, and the Champion Reverse Weave Hoodie for $59.99. Which one would you like?"}, {'role': 'user', 'text': ' Okay second one.'}, {'role': 'ai', 'text': 'Shall I add the Champion Reverse Weave Hoodie to your cart?'}, {'role': 'user', 'text': ' E ele'}, {'role': 'ai', 'text': "I've added the Champion Reverse Weave Hoodie to your cart. Is there anything else I can help you with today?"}, {'role': 'user', 'text': ' what is a very'}, {'role': 'ai', 'text': "I'm sorry, I didn't find any products matching 'belt'. Are you looking for something specific, like a leather belt?"}, {'role': 'user', 'text': ' electronics. Can you hear me?'}, {'role': 'ai', 'text': 'Yes, I can hear you clearly. Did you want to search for "electronics"?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': 'I\'m sorry, I didn\'t find any products matching "electronics". We might not have that category. Are you looking for something specific like "headphones" or "charger"?'}, {'role': 'user', 'text': ' Yes what is headphone?'}, {'role': 'ai', 'text': 'I found two products: Sony WH-1000XM5 for $349.99, and Apple AirPods Pro (2nd Gen) for $249.99. Which one are you interested in?'}, {'role': 'user', 'text': " Okay, yeah, let's go to Sony."}, {'role': 'ai', 'text': 'Would you like to add the Sony WH-1000XM5 to your cart?'}, {'role': 'user', 'text': ' Yes.'}, {'role': 'ai', 'text': "I've added the Sony headphones to your cart. Is there anything else?"}, {'role': 'user', 'text': 'What about jeans'}, {'role': 'ai', 'text': "I found Levi's 501 Original Jeans for $69.99. Are you interested in these?"}, {'role': 'user', 'text': 'Good add it'}, {'role': 'ai', 'text': "Shall I add the Levi's 501 Original Jeans to your cart?"}, {'role': 'user', 'text': 'Ues and add 2 of it'}, {'role': 'ai', 'text': "I've added 2 quantities of the Levi's 501 Jeans to your cart. Is there anything else I can help you with?"}, {'role': 'user', 'text': ' Wat een kracht.'}, {'role': 'ai', 'text': "Your cart contains: 1 Champion Reverse Weave Hoodie, 1 Sony WH-1000XM5 and 2 Levi's 501 Original Jeans. The total is $549.96. Ready to checkout?"}]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\core\voice\gemini_live_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-26T20:16:56.306203Z [info     ] gemini_session_closing         [voice_commerce.core.voice.gemini_live_handler]
INFO:     Waiting for background tasks to complete. (CTRL+C to force quit)
2026-03-26T20:16:56.486442Z [info     ] gemini_session_closed_by_server [voice_commerce.core.voice.gemini_live_handler] reason='1000 None. '
2026-03-26T20:16:56.486442Z [info     ] gemini_session_closed_gracefully [voice_commerce.handlers.voice_websocket_handler] reason='1000 None. ' session=v5_mph3j0z2
2026-03-26T20:16:56.488381Z [info     ] gemini_session_closed          [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:16:56.488381Z [info     ] ws_disconnected_normally       [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 turns=34
2026-03-26T20:16:56.489997Z [info     ] ws_handler_finished            [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 total_turns=34
INFO:     Waiting for application shutdown.
2026-03-26T20:16:56.523596Z [info     ] Application shutting down      [src.voice_commerce.main]
INFO:     Application shutdown complete.
INFO:     Finished server process [19312]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\core\voice\gemini_live_handler.py'. Reloading...
 INFO:     Started server process [22228]
INFO:     Waiting for application startup.
2026-03-26T20:16:59.612522Z [info     ] voice_commerce_agent_starting  [src.voice_commerce.main] debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-26T20:16:59.785678Z [info     ] Application shutting down      [src.voice_commerce.main]
INFO:     Application shutdown complete.
INFO:     Finished server process [22228]
INFO:     Started server process [21252]
INFO:     Waiting for application startup.
2026-03-26T20:17:02.448424Z [info     ] voice_commerce_agent_starting  [src.voice_commerce.main] debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-26T20:17:02.533355Z [info     ] voice_handler_created          [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
INFO:     127.0.0.1:2030 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-26T20:17:02.533355Z [info     ] ws_connected                   [voice_commerce.handlers.voice_websocket_handler] client="Address(host='127.0.0.1', port=2030)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-26T20:17:02.704288Z [info     ] gemini_session_connecting      [voice_commerce.core.voice.gemini_live_handler] model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-26T20:17:03.857472Z [info     ] gemini_session_connected       [voice_commerce.core.voice.gemini_live_handler]
2026-03-26T20:17:09.019225Z [info     ] input_mode_switched_to_audio   [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2
2026-03-26T20:17:24.094900Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'ai', 'text': 'Sure, I can help with that. What product ID should I remove from your cart?'}]
2026-03-26T20:17:27.494215Z [info     ] user_speech_transcribed        [voice_commerce.handlers.voice_websocket_handler] session_id=v5_mph3j0z2 text=' in my car now.'
2026-03-26T20:17:27.769044Z [info     ] tool_call_received             [voice_commerce.handlers.voice_websocket_handler] args={} call_id=function-call-11513857942445210840 session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:17:27.769044Z [info     ] Executing tool                 [voice_commerce.core.tools.tool_dispatcher] arguments={} session=v5_mph3j0z2 tool_name=show_cart
2026-03-26T20:17:27.769044Z [info     ] dispatcher_calling             [voice_commerce.core.tools.tool_dispatcher] args={} session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:17:27.769044Z [info     ] show_cart                      [voice_commerce.core.tools.cart_tools] session=v5_mph3j0z2
2026-03-26T20:17:27.770649Z [info     ] dispatcher_done                [voice_commerce.core.tools.tool_dispatcher] result_len=65 tool=show_cart
2026-03-26T20:17:27.770649Z [info     ] tool_call_result               [voice_commerce.handlers.voice_websocket_handler] preview='Your cart is empty. Would you like me to help you find something?' session=v5_mph3j0z2 tool=show_cart
2026-03-26T20:17:27.770649Z [info     ] what we can do                 [voice_commerce.core.voice.gemini_live_handler] args={} call_id=function-call-11513857942445210840 function_call=FunctionCall(
  args={},
  id='function-call-11513857942445210840',
  name='show_cart'
)
2026-03-26T20:17:31.922929Z [info     ] memory                         [voice_commerce.handlers.voice_websocket_handler] memory=[{'role': 'ai', 'text': 'Sure, I can help with that. What product ID should I remove from your cart?'}, {'role': 'user', 'text': ' in my car now.'}, {'role': 'ai', 'text': 'Your cart is empty. Would you like me to help you find something?'}]


In [ ]:
import asyncio
import os
import logging 
from unittest import result
from google import genai
from google.genai import types
# 1. Define a tool the model can call
log = logging.getLogger(__name__)
def get_current_time():
    """Returns the current system time."""
    from datetime import datetime
    return {"time": datetime.now().strftime("%H:%M:%S")}

# 2. Setup Client
# Ensure you use 'v1alpha' for preview features
client = genai.Client(
    api_key="YOUR_API_KEY", 
    http_options={'api_version': 'v1alpha'}
)

MODEL_ID = "gemini-2.5-flash-native-audio-preview-12-2025"

async def main():
    # 3. Configure the session for audio + text + tools
    config = types.LiveConnectConfig(
        tools=[get_current_time],
        response_modalities=["AUDIO"], # Returns native audio
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        # This returns the text transcript of the model's audio response
        output_audio_transcription=types.AudioTranscriptionConfig()
    )

    async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
        # await session.send_tool_response()
        # Send text input or audio bytes (e.g., from a mic stream)
        # To send audio, replace the string with audio bytes:
        # await session.send(audio_bytes, end_of_turn=True)
        await session.send_client_content(turns= types.Content(role="user", parts=[types.Part(text="What time is it?")]), turn_complete=True)

        try:
            # FIXED: We open the Google stream EXACTLY ONCE and listen forever.
            # No 'while True:' needed, because 'async for' naturally waits for new data.
            async for response in session.receive():
                
                server_content = getattr(response, "server_content", None)
                if not server_content:
                    continue

                # 1. Handle Text, Audio, and Tools
                if server_content.model_turn and server_content.model_turn.parts:
                    for part in server_content.model_turn.parts:
                        
                        if getattr(part, "text", None):
                            log.debug("gemini_text_chunk", length=len(part.text))
                            yield {"type": "text", "text": part.text}

                        if getattr(part, "inline_data", None):
                            log.debug("gemini_audio_chunk", bytes=len(part.inline_data.data))
                            yield {"type": "audio", "data": part.inline_data.data}
                        
                        if getattr(part, "function_call", None):
                            log.info("gemini_tool_call_detected", tool_name=part.function_call.name)
                            yield {
                                "type": "tool_call",
                                "name": part.function_call.name,
                                "args": dict(part.function_call.args or {}),
                                "call_id": getattr(part.function_call, "id", None),
                            }

                # 2. Handle Transcripts (What you said, and what it heard)
                input_trans = getattr(server_content, "input_transcription", None)
                if input_trans and getattr(input_trans, "text", None):
                    log.debug("gemini_input_transcript", text=input_trans.text[:80])
                    yield {"type": "input_transcript", "text": input_trans.text}

                output_trans = getattr(server_content, "output_transcription", None)
                if output_trans and getattr(output_trans, "text", None):
                    log.debug("gemini_output_transcript", text=output_trans.text[:80])
                    yield {"type": "output_transcript", "text": output_trans.text}

                # 3. ── THE MAGIC FIX ──
                # Tell the UI the AI is done speaking so the mic unlocks, 
                # but DO NOT BREAK! The line stays open for the next text or voice input.
                if server_content.turn_complete:
                    yield {"type": "turn_complete"}

        except Exception as exc:
            error_str = str(exc)
            if "1000" in error_str or "1001" in error_str:
                log.info("gemini_session_closed_by_server", reason=error_str)
                yield {"type": "session_closed", "reason": error_str}
                return
            log.error("gemini_receive_error", error=error_str)
            yield {"type": "error", "message": error_str}
            return
        

                     

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
import time
print(time.time())

In [3]:
l = [1,2,3,4,5]
print(l[:2])

[1, 2]


In [ ]:
from __future__ import annotations
from  collections.abc import AsyncGenerator
from typing import Any


import structlog
from google import genai
from google.genai import types

from voice_commerce.core.tools import tool_registry   # ← Phase 3: import the tool declarations
from voice_commerce.core.voice import audio_processor  # ← Phase 5: import audio config for browser
from voice_commerce.config.settings import settings


log = structlog.get_logger(__name__)


class GeminiLiveHandler:

    def __init__(self) -> None:
        self._client = genai.Client(api_key=settings.gemini_api_key) 
        self._session: Any = None
        self._session_ctx: Any = None   
        self._config = self._build_session_config()

        log.debug(
            "gemini_live_handler_created",
            model=settings.gemini_model,
            voice=settings.gemini_voice_name,
        )

    def _build_session_config(self) -> types.LiveConnectConfig:
        """Builds the configuration for the Gemini speech session."""
        

        return types.LiveConnectConfig(

            response_modalities=[types.Modality.AUDIO],
            system_instruction=types.Content(
                parts=[
                    types.Part(
                        text=self._build_system_prompt()
                    )
                ]
            ),
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name=settings.gemini_voice_name
                        # Available: "Aoede", "Charon", "Fenrir", "Kore", "Puck"
                        # Charon: clear, neutral — good for shopping assistant
                    )
                )
            ),
            input_audio_transcription=types.AudioTranscriptionConfig(),   
            output_audio_transcription=types.AudioTranscriptionConfig(),
            tools=tool_registry.get_all_tools(),   # ← Phase 3: the one new line

            )
            

    
    def _build_system_prompt(self) -> str:
        """The 'Rules' the AI must follow."""
        return (
            "You are a friendly Voice Shopping Assistant. "
            "ALWAYS respond in English only, regardless of the language the user speaks. "
            "Keep answers short and conversational — 1 to 3 sentences maximum. "
            "If you don't understand the user, ask one short clarifying question."
        )
    

    async def send_text(self, text: str) -> None:
        """Sends text to Gemini and yields response text as it arrives."""
        if self._session is None:
            raise RuntimeError(
                "Cannot send text: Gemini session is not connected. "
                "Did you use 'async with handler.connect():'?"
            )
        
        log.debug("gemini_sending_text", text=text)

        await self._session.send_client_content(
            turns=types.Content(
                role="user",
                parts=[types.Part(text=text)]
            ),
            turn_complete=True,
            # turn_complete=True is the key flag that tells Gemini:
            # "The user is done speaking, please respond now."
        )



    async def send_audio_chunk(self, pcm_bytes: bytes) -> None:
        """
        Stream mic audio from the browser to Gemini.
        Added in Phase 5 (microphone input). Kept here ready for that phase.
 
        Format required: PCM s16le at 16kHz mono.
        Uses send_realtime_input() not send_client_content() because audio is
        a continuous stream — Gemini's VAD detects speech pauses automatically.
        """
        log.debug("gemini_sending_audio_chunk", bytes=len(pcm_bytes))
        if self._session is None:
            raise RuntimeError("Cannot send audio: session not connected.")
        if not pcm_bytes:
            return  # skip empty chunks (can happen with some mic implementations)

        await self._session.send_realtime_input(
            audio=types.Blob(
                data=pcm_bytes,
                mime_type=f"audio/pcm;rate={audio_processor.MIC_SAMPLE_RATE}",
            )
        )


    async def send_tool_result(self, call_id:   str | None, tool_name: str,result:    str) -> None:

        if self._session is None:
            raise RuntimeError("Cannot send tool result: not connected.")
        log.debug("gemini_sending_tool_result",tool=tool_name, preview=result[:80])

        await self._session.send_tool_response(
            function_responses=[
                types.FunctionResponse(
                    id=call_id,  # correlate with the tool call
                    name=tool_name,
                    response={"result": result},
                )
            ],
        )

            # """
            # Send a tool's return value back to Gemini after executing it.  ← new Phase 3
 
            # FULL FLOW:
            # 1. User:  "show me running shoes"
            # 2. Gemini yields tool_call → name="search_products", args={"query":"running shoes"}
            # 3. Handler dispatches → search_products() runs → returns result string
            # 4. Handler calls this method → result delivered to Gemini
            # 5. Gemini reads result → speaks it naturally to the user
    
            # WHY send_tool_response() NOT send_client_content():
            # send_client_content() is for USER messages (text or audio the user sent).
            # send_tool_response() is for FUNCTION RESULTS.
            # The Gemini API routes them to completely different internal processors.
            # Using send_client_content() here would make Gemini treat your search
            # results as if the USER said them — broken conversation structure.
    
            # WHY result is a string (not JSON or a dict):
            # Gemini reads the result as text context, then paraphrases it into
            # natural speech. A structured readable string like:
            #     "• Nike Air Zoom — $129 — ID:1\n  Lightweight running shoe."
            # is ideal. Gemini turns this into:
            #     "I found the Nike Air Zoom for a hundred and twenty-nine dollars.
            #     It's a lightweight running shoe. Want to add it to your cart?"
    
            # ABOUT call_id:
            # Gemini assigns a unique ID to each tool call for correlation.
            # It can be None for some model versions — send_tool_response handles None.
            # """

    async def receive_events(self) -> AsyncGenerator[dict[str, Any], None]:
        if self._session is None:
            raise RuntimeError("Cannot receive: Gemini session is not connected.")
        while True:
            try:
                async for response  in self._session.receive():
                    # Gemini sends text in chunks (like a typewriter)
                    response: types.LiveServerMessage
                    if response.text is not None:
                        log.debug("gemini_text_chunk", length=len(response.text))
                        yield {"type": "text", "text": response.text}

                    if response.data is not None:
                        log.debug("gemini_audio_chunk", bytes=len(response.data))
                        yield {"type": "audio", "data": response.data}
                    
                    # ── Tool call ← Phase 3 ─────────────────────────────────
                    # When Gemini decides to call a function, it sends the call
                    # in model_turn.parts (not in response.text or response.data).
                    # We detect it here and yield it so the handler can dispatch it.
                    #
                    # WHY check model_turn.parts (not just response.text):
                    #   Function calls are a separate part type in the Gemini API.
                    #   They arrive as Part objects with a function_call field,
                    #   not as text. You cannot detect them from response.text.

                    if (
                        response.server_content
                        and response.server_content.model_turn
                        and response.server_content.model_turn.parts
                    ):
                        for part in response.server_content.model_turn.parts:
                            if hasattr(part, "function_call") and part.function_call:
                                log.info("gemini_tool_call_detected",tool_name=part.function_call.name)
                                yield {
                                    "type": "tool_call",
                                    "name": part.function_call.name,
                                    "args": dict(part.function_call.args or {}),
                                    "call_id": getattr(part.function_call, "id", None),
                                }

                    # INPUT TRANSCRIPT (Phase 5+)
                    # Text version of what the USER said (from their audio)
                    if (
                        response.server_content
                        and hasattr(response.server_content, "input_transcription")
                        and response.server_content.input_transcription
                    ):
                        transcript = response.server_content.input_transcription
                        if hasattr(transcript, "text") and transcript.text:
                            log.debug("gemini_input_transcript", text=transcript.text[:80])
                            yield {"type": "input_transcript", "text": transcript.text}
    
                    # OUTPUT TRANSCRIPT (Phase 4+)
                    # Text version of what GEMINI said (from its audio response)
                    if (
                        response.server_content
                        and hasattr(response.server_content, "output_transcription")
                        and response.server_content.output_transcription
                    ):
                        transcript = response.server_content.output_transcription
                        if hasattr(transcript, "text") and transcript.text:
                            log.debug("gemini_output_transcript", text=transcript.text[:80])
                            yield {"type": "output_transcript", "text": transcript.text}

                    # Check if Gemini is finished with its thought
                    if response.server_content and response.server_content.turn_complete:
                        yield {"type": "turn_complete"}
                        break  # exit inner loop → outer loop restarts immediately
            except Exception as exc:
                error_str = str(exc)
                if "1000" in error_str or error_str.strip() in ("1000 None.", "1001 None."):
                    log.info("gemini_session_closed_by_server", reason=error_str)
                    yield {"type": "session_closed", "reason": error_str}
                    return  # Stop — session is over, browser will auto-reconnect
                # Real error — log, notify browser, stop
                log.error("gemini_receive_error", error=error_str)
                yield {"type": "error", "message": error_str}
                return


    async def __aenter__(self) -> "GeminiLiveHandler":
        log.info("gemini_session_connecting", model=settings.gemini_model   )

        self._session_ctx = self._client.aio.live.connect(
            model=settings.gemini_model,
            config=self._config,
        )

        # Enter the context manager → opens the WebSocket to Gemini's servers
        self._session = await self._session_ctx.__aenter__()
        log.info("gemini_session_connected")
        return self
    

    async def __aexit__(self, exc_type: Any, exc_val: Any, exc_tb: Any) -> None:
        """ 
        Close the Gemini Live session.
        Called automatically when exiting the `async with` block.
 
        exc_type, exc_val, exc_tb: exception info if an exception occurred
        (None, None, None if the block exited normally)
        """
        log.info("gemini_session_closing")
 
        if self._session_ctx is not None:
            try:
                # Close the google-genai context manager (closes the WebSocket)
                await self._session_ctx.__aexit__(exc_type, exc_val, exc_tb)
            except Exception as e:
                # Don't let cleanup errors mask the original exception
                log.warning("gemini_session_close_error", error=str(e))
            finally:
                self._session = None                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          
                self._session_ctx = None
 
        log.info("gemini_session_closed")

2026-03-24 07:20:12 [info     ] browser_message_received       message={'type': 'websocket.receive', 'text': '{"type":"text","text":"Need summar teshirt"}'} message_type=websocket.receive session_id=v5_mph3j0z2

2026-03-24 07:18:12 [info     ] browser_message_received       message={'type': 'websocket.receive', 'bytes': b'\x9c\xff\x8d\xff\x81\xffi\xffX\xff<\xff\x19\xff\x02\xff\xf0\xfe\xe3\xfe\xe3\xfe\xf0\xfe\x13\xff=\xff`\xff\x9a\xff\xcd\xff\x08\x00E\x00p\x00\x99\x00\xb5\x00\xc8\x00\xd7\x00\xef\x00\x12\x015\x01c\x01\x91\x01\xb1\x01\xc4\x01\xc4\x01\xac\x01\x7f\x01K\x01\x05\x01\xca\x00\x9a\x00n\x00P\x00-\x00\x0f\x00\xfd\xff\xf0\xff\xe6\xff\xcf\xff\xa7\xffm\xff,\xff\xe6\xfe\xb9\xfe\xa4\xfe\xa8\xfe\xbe\xfe\xca\xfe\xdc\xfe\xf7\xfe\x11\xff/\xffS\xff{\xff\xa1\xff\xc7\xff\xea\xff\x11\x00D\x00s\x00\xa0\x00\xd4\x00\x06\x012\x01J\x01L\x01@\x01/\x01\x0b\x01\xe7\x00\xcb\x00\xa7\x00\x8f\x00w\x00g\x00Z\x00E\x00.\x00\x02\x00\xd5\xff\xa7\xff\x88\xffm\xffS\xffC\xff-\xff\x10\xff\xf4\xfe\xe9\xfe\xe2\xfe\xe1\xfe\xde\xfe\xd8\xfe\xdc\xfe\xed\xfe\x13\xff9\xffa\xff\x90\xff\xb8\xff\xdc\xff\x07\x007\x00t\x00\xb0\x00\xdd\x00\x15\x01E\x01X\x01k\x01q\x01^\x01J\x01C\x011\x01#\x01\x1e\x01\x0c\x01\xeb\x00\xc4\x00\x9c\x00u\x00M\x00:\x00\x18\x00\xf1\xff\xd0\xff\x8e\xffT\xff"\xff\xdf\xfe\xab\xfe\x83\xfeY\xfe-\xfe\r\xfe\x05\xfe\x03\xfe\x1d\xfe0\xfe>\xfeZ\xfeq\xfe\x91\xfe\xb7\xfe\xde\xfe\x1b\xffo\xff\xc3\xff\x1f\x00u\x00\xbe\x00\xf4\x00\x15\x013\x01M\x01c\x01z\x01\x84\x01\x84\x01\x9e\x01\xae\x01\xb2\x01\xb1\x01\x92\x01f\x018\x01\xf7\x00\xb1\x00v\x00*\x00\xde\xff\xa2\xffn\xffM\xffC\xff.\xff\x11\xff\xf1\xfe\xd9\xfe\xc2\xfe\xa8\xfe\x89\xfek\xfeN\xfe7\xfe&\xfe2\xfeW\xfe\x80\xfe\xad\xfe\xe1\xfe\xfe\xfe\x0e\xff\x1e\xff%\xff>\xffp\xff\xb6\xff\n\x00q\x00\xd5\x00+\x01o\x01\x92\x01\x9b\x01\x92\x01{\x01M\x011\x01\x14\x01\n\x01\x13\x01+\x01K\x01W\x01M\x01+\x01\xf5\x00\xb0\x00k\x00>\x00\x17\x00\xed\xff\xd4\xff\xbe\xff\x91\xffb\xff/\xff\xe9\xfe\xab\xfe\x84\xfel\xfec\xfeo\xfe\x86\xfe\xa3\xfe\xd3\xfe\x10\xffK\xff\x93\xff\xce\xff\xf8\xff\x19\x00/\x00C\x00W\x00v\x00\x88\x00\xa6\x00\xd2\x00\xe5\x00\xfd\x00\x16\x01*\x01C\x01C\x01;\x01+\x01\x0b\x01\xe6\x00\xc8\x00\xb1\x00\x9c\x00\x8c\x00\x7f\x00o\x00X\x00G\x00\x14\x00\xd7\xff\xa3\xffj\xff?\xff$\xff\x13\xff\x03\xff\x04\xff\t\xff\x11\xff(\xff8\xff2\xff7\xff;\xffQ\xfft\xff\x9c\xff\xc3\xff\xe7\xff\x16\x00E\x00k\x00\x95\x00\xb7\x00\xc5\x00\xc3\x00\xba\x00\xaf\x00\xaf\x00\xb5\x00\xc5\x00\xe2\x00\xfa\x00\x08\x01\x13\x01\x12\x01\xfa\x00\xe6\x00\xc7\x00\x98\x00{\x00Y\x001\x00\x17\x00\xf5\xff\xe6\xff\xd7\xff\xb6\xff\x96\xfff\xff/\xff\xfa\xfe\xd3\xfe\xbc\xfe\xba\xfe\xd9\xfe\x04\xff5\xff}\xff\xc1\xff\xeb\xff\xf8\xff\xf9\xff\xec\xff\xdb\xff\xda\xff\xe3\xff\xf0\xff\x0e\x00;\x00]\x00\x8b\x00\xad\x00\xc3\x00\xd0\x00\xc3\x00\xb1\x00\x97\x00\x82\x00\x80\x00\x83\x00\x93\x00\xaa\x00\xb8\x00\xb9\x00\xb2\x00\x9a\x00w\x00E\x00\x12\x00\xd2\xff\x8e\xff^\xff6\xff#\xff!\xff+\xff1\xff$\xff\x13\xff\xf8\xfe\xde\xfe\xcb\xfe\xd2\xfe\xed\xfe\xfe\xfe\x1f\xffJ\xffx\xff\xb2\xff\xfb\xff)\x00P\x00q\x00x\x00o\x00s\x00\x8b\x00\x91\x00\xa4\x00\xbc\x00\xcf\x00\xe9\x00\xff\x00\x0c\x01\x06\x01\xf9\x00\xe1\x00\xb2\x00\x8d\x00g\x00=\x00\x19\x00\xf8\xff\xe3\xff\xbd\xff\x9f\xff}\xffH\xff\x10\xff\xe6\xfe\xc7\xfe\xa5\xfe\x91\xfe\x8e\xfe\x8c\xfe\x94\xfe\xad\xfe\xce\xfe\xf2\xfe\x19\xff9\xffR\xffg\xff}\xff\x99\xff\xb9\xff\xe6\xff\x10\x008\x00_\x00\x89\x00\xa9\x00\xc0\x00\xd6\x00\xe3\x00\xe9\x00\xe7\x00\xe8\x00\xef\x00\xef\x00\xed\x00\xec\x00\xde\x00\xd3\x00\xc7\x00\xa8\x00\x8d\x00h\x006\x00\x02\x00\xd7\xff\xb9\xff\x9c\xff\x86\xff\x83\xffp\xff_\xffQ\xff<\xff!\xff\x04\xff\xe6\xfe\xc9\xfe\xbb\xfe\xc4\xfe\xd0\xfe\xfa\xfe+\xffS\xffs\xff\x9e\xff\xcd\xff\xe5\xff\xfb\xff\x13\x00+\x00L\x00w\x00\xad\x00\xea\x00\x1d\x01L\x01n\x01z\x01\x83\x01\x86\x01s\x01]\x01G\x015\x012\x01*\x01\x19\x01\x03\x01\xe4\x00\xb6\x00\x85\x00A\x00\x05\x00\xc6\xff\x80\xffV\xff4\xff\x1a\xff\x15\xff\n\xff\xf5\xfe\xe1\xfe\xcc\xfe\xbf\xfe\xb5\xfe\xb1\xfe\xb6\xfe\xc2\xfe\xd6\xfe\xf3\xfe\x16\xff?\xffb\xff\x8f\xff\xc2\xff\xed\xff\x1f\x00N\x00o\x00\x8d\x00\xb8\x00\xea\x00(\x01\\\x01z\x01\x85\x01\x7f\x01l\x01\\\x01W\x01P\x01C\x011\x01\x1e\x01\x06\x01\xe8\x00\xc2\x00\x98\x00a\x00 \x00\xea\xff\xb3\xff\x84\xffd\xffD\xff*\xff\x19\xff\x03\xff\xf1\xfe\xd5\xfe\xb6\xfe\xae\xfe\xb2\xfe\xb7\xfe\xcd\xfe\xdf\xfe\xff\xfe#\xffO\xff\x80\xff\xa9\xff\xd5\xff\xf2\xff\x07\x00\x19\x00?\x00i\x00\x89\x00\xb7\x00\xda\x00\xf4\x00\r\x01%\x011\x01;\x01H\x01B\x01-\x01\x11\x01\xf6\x00\xd3\x00\xb8\x00\xa4\x00\x84\x00_\x00;\x00\t\x00\xd1\xff\xab\xff\x84\xffP\xff\x1a\xff\xda\xfe\xb0\xfe\x8d\xfez\xfe}\xfe\x7f\xfe\x83\xfe\x83\xfez\xfe~\xfe\x8b\xfe\xa0\xfe\xcc\xfe\xfc\xfe&\xff\\\xff\x8a\xff\xc7\xff\xfa\xff,\x00b\x00\x91\x00\xc4\x00\xe0\x00\xff\x00\x19\x01 \x01\x1f\x01\x1d\x01%\x01\'\x01,\x01\x1e\x01\x11\x01\x03\x01\xd8\x00\xa9\x00e\x000\x00\x01\x00\xd2\xff\xaa\xffu\xffL\xff\x1e\xff\x05\xff\xeb\xfe\xd8\xfe\xd3\xfe\xb9\xfe\xa5\xfe\x91\xfe\x92\xfe\xb6\xfe\xfc\xfeU\xff\xaf\xff\xf6\xff!\x005\x004\x00(\x00 \x00\r\x00\x02\x00\x0b\x00$\x00L\x00\x86\x00\xc6\x00\xf9\x00\x1c\x013\x01=\x01>\x01;\x01-\x01\x1b\x01\n\x01\xeb\x00\xd0\x00\xbe\x00\xa8\x00\x89\x00d\x00&\x00\xdb\xff\x98\xff_\xff7\xff\x1f\xff \xff+\xff4\xff>\xffB\xffB\xffA\xff>\xff1\xff1\xff5\xff<\xffV\xff\x85\xff\xc0\xff\x02\x00I\x00\x84\x00\xbf\x00\xde\x00\xed\x00\xf0\x00\xeb\x00\xeb\x00\xe7\x00\xe1\x00\xd2\x00\xc3\x00\xb8\x00\xad\x00\x9f\x00\x86\x00m\x00T\x00C\x005\x00,\x00(\x00.\x005\x000\x00\x1d\x00\xff\xff\xda\xff\xa3\xffY\xff\x1a\xff\xcf\xfe\x97\xfeu\xfeb\xfee\xfe{\xfe\xa7\xfe\xde\xfe\x1f\xffZ\xff\x97\xff\xc9\xff\xf8\xff"\x00J\x00\x7f\x00\xa8\x00\xc0\x00\xda\x00\xe9\x00\xf1\x00\x01\x01\x07\x01\x08\x01\x08\x01\xf8\x00\xe6\x00\xc7\x00\xac\x00\x9b\x00\x90\x00\x91\x00\x9f\x00\xb5\x00\xbb\x00\xbf\x00\xad\x00{\x00D\x00\x02\x00\xc0\xff\x80\xffL\xff\x1b\xff\xf6\xfe\xd5\xfe\xb4\xfe\x98\xfe\x81\xfey\xfet\xfep\xfei\xfeb\xfeq\xfe\x8b\xfe\xc4\xfe\x08\xffM\xff\x9d\xff\xc9\xff\xe3\xff\xf2\xff\xe8\xff\xf0\xff\xf4\xff\x01\x00(\x00U\x00\x81\x00\xb5\x00\xe3\x00\x0e\x01$\x01)\x01\'\x01\x19\x01\x08\x01\xfa\x00\xed\x00\xe8\x00\xef\x00\x02\x01\x13\x01\x1e\x01\x16\x01\xf4\x00\xb8\x00e\x00\x12\x00\xb4\xffg\xff2\xff\x0b\xff\x00\xff\xf8\xfe\xec\xfe\xe2\xfe\xe8\xfe\xe1\xfe\xd7\xfe\xdd\xfe\xe1\xfe\xf6\xfe\xfd\xfe\x0b\xff)\xffW\xff\x88\xff\xc3\xff\x00\x00\'\x00A\x00T\x00f\x00\x85\x00\xa9\x00\xdc\x00\x13\x01O\x01\x84\x01\xae\x01\xc4\x01\xbe\x01\xb0\x01\x8d\x01d\x014\x01\xf6\x00\xb9\x00\x83\x00W\x00\'\x00\x02\x00\xe3\xff\xcc\xff\xb6\xff\x95\xffi\xff0\xff\x04\xff\xe2\xfe\xca\xfe\xb7\xfe\xbf\xfe\xd0\xfe\xda\xfe\xe4\xfe\xf2\xfe\xf7\xfe\x03\xff\r\xff\x19\xff4\xffW\xff\x81\xff\xa8\xff\xd8\xff\x02\x00:\x00y\x00\xad\x00\xe1\x00\x03\x01\x07\x01\xf1\x00\xdc\x00\xd0\x00\xdb\x00\xf8\x00\x0f\x01\x1e\x01\x17\x01\x00\x01\xe0\x00\xba\x00\x9b\x00\x80\x00U\x00\x1b\x00\xde\xff\xa4\xff\x85\xffv\xffr\xff\x81\xff\x82\xff|\xffm\xff]\xffK\xff)\xff\x01\xff\xec\xfe\xd2\xfe\xc8\xfe\xdb\xfe\xe6\xfe\x04\xff&\xff2\xffC\xffW\xffg\xff\x89\xff\xaf\xff\xda\xff\x01\x00#\x00J\x00h\x00\x8e\x00\xbb\x00\xdc\x00\xf5\x00\x04\x01\x0c\x01\x0e\x01\t\x01\x07\x01\x0f\x01\x1c\x01\x1c\x01\t\x01\xe0\x00\xb1\x00~\x00E\x00\x15\x00\xe6\xff\xc4\xff\xb4\xff\xb6\xff\xae\xff\xbe\xff\xd2\xff\xc8\xff\xca\xff\xbd\xff\xaf\xff\xa8\xff\x96\xff\x7f\xff^\xffD\xff6\xff5\xffH\xff\x81\xff\xad\xff\xcd\xff\xe1\xff\xe3\xff\xe9\xff\xfa\xff\x12\x008\x00b\x00\x8d\x00\x9c\x00\xa5\x00\xb8\x00\xba\x00\xb8\x00\xac\x00\x93\x00t\x00c\x00M\x00=\x007\x00+\x00+\x00\x1e\x00\x0f\x00\xfa\xff\xe0\xff\xbf\xff\x92\xff}\xffq\xffd\xffU\xffW\xff[\xff]\xffq\xff\x84\xff\x92\xff\xa4\xff\xaa\xff\xab\xff\xac\xff\xbe\xff\xd6\xff\xf2\xff\t\x00*\x00D\x00S\x00k\x00\x7f\x00\x93\x00\x99\x00\x9a\x00\x92\x00\x93\x00\x97\x00\x94\x00\x9f\x00\xaa\x00\xb0\x00\xb4\x00\xb7\x00\xab\x00\x9f\x00\x80\x00H\x00\x0c\x00\xcf\xff\x8d\xffa\xff?\xff!\xff\x04\xff\xe2\xfe\xc4\xfe\xa7\xfe\x98\xfe\x94\xfe\x9a\xfe\xac\xfe\xc9\xfe\xec\xfe\xfe\xfe\x1e\xff1\xffI\xffo\xff\x91\xff\x9e\xff\xaf\xff\xc0\xff\xc0\xff\xcf\xff\xef\xff&\x00f\x00\xa8\x00\xf0\x001\x01n\x01\x97\x01\xb5\x01\xd4\x01\xda\x01\xdc\x01\xcc\x01\xb0\x01\x8d\x01Z\x01\x1c\x01\xe1\x00\xaf\x00w\x00G\x00\x19\x00\xe9\xff\xb9\xff\x8e\xffd\xffQ\xff?\xff8\xff<\xff0\xff&\xff\x18\xff\xfe\xfe\xf2\xfe\xe9\xfe\xe3\xfe\xe8\xfe\xf1\xfe\x00\xff\x1f\xffE\xffv\xff\xae\xff\xed\xff3\x00m\x00\x99\x00\xc7\x00\xf2\x00\x11\x01*\x01>\x01H\x01M\x01R\x01S\x01F\x014\x01\x1e\x01\xfe\x00\xe0\x00\xba\x00\x93\x00s\x00]\x00I\x003\x00\x16\x00\xf2\xff\xd1\xff\xb0\xff\x98\xffv\xffG\xff\x18\xff\xee\xfe\xc7\xfe\xa3\xfe\x94\xfe\x97\xfe\x9d\xfe\x9d\xfe\xa5\xfe\x9d\xfe\xa2\xfe\xb7\xfe\xd1\xfe\x02\xff3\xfff\xff\x95\xff\xce\xff\t\x00B\x00z\x00\xa9\x00\xd3\x00\xf3\x00\x14\x014\x01K\x01\\\x01b\x01s\x01p\x01i\x01^\x01B\x01"\x01\x02\x01\xe2\x00\xb6\x00\x84\x00_\x003\x00\n\x00\xe8\xff\xc1\xff\x8c\xffN\xff\x1d\xff\xf0\xfe\xd2\xfe\xbb\xfe\xa5\xfe\x93\xfe\x86\xfex\xfen\xfen\xfey\xfe\x9e\xfe\xc0\xfe\xe4\xfe\x11\xff1\xff`\xff\x99\xff\xd4\xff\x07\x00D\x00x\x00\x9f\x00\xcb\x00\xfe\x00,\x01P\x01o\x01x\x01x\x01|\x01p\x01b\x01\\\x01J\x01#\x01\xf2\x00\xba\x00\x86\x00X\x00)\x00\x00\x00\xd2\xff\xa7\xff\x81\xff^\xffK\xff0\xff\n\xff\xf7\xfe\xdf\xfe\xc2\xfe\xb3\xfe\x9f\xfe\x90\xfe\x93\xfe\x9f\xfe\xb0\xfe\xc5\xfe\xd9\xfe\xf0\xfe\x14\xffC\xffl\xff\x9b\xff\xca\xff\xf1\xff\x14\x00=\x00m\x00\x98\x00\xc0\x00\xda\x00\xe8\x00\xec\x00\xe3\x00\xd3\x00\xb5\x00\xa4\x00\x9a\x00\x8b\x00|\x00j\x00Z\x00@\x00,\x00\t\x00\xea\xff\xd6\xff\xb9\xff\x90\xffs\xffe\xffR\xffE\xff4\xff%\xff\x1c\xff\x16\xff\n\xff\xff\xfe\x00\xff\xfd\xfe\x06\xff\x18\xff,\xffM\xffo\xff\x91\xff\xb6\xff\xd4\xff\xfd\xff\'\x00L\x00u\x00\xa0\x00\xb7\x00\xd5\x00\xf2\x00\x01\x01\x13\x01&\x018\x016\x01A\x01F\x01=\x01/\x01\x15\x01\xf7\x00\xd3\x00\xac\x00\x86\x00d\x009\x00\x0c\x00\xe5\xff\xc4\xff\xa5\xff\x7f\xff_\xff>\xff\x1a\xff\x05\xff\xf3\xfe\xe9\xfe\xf5\xfe\x0f\xff-\xffR\xff\x7f\xff\xa4\xff\xc0\xff\xde\xff\x01\x00!\x00F\x00l\x00\x93\x00\xc2\x00\xea\x00\x13\x017\x01O\x01Y\x01X\x01L\x01@\x01,\x01\x16\x01\xfe\x00\xef\x00\xec\x00\xde\x00\xcd\x00\xb3\x00\x88\x00W\x00$\x00\xf9\xff\xd9\xff\xb8\xff\xa2\xff\x8b\xffi\xffP\xff8\xff,\xff&\xff\x1f\xff\x1a\xff\x0f\xff\x01\xff\xfe\xfe\x10\xff%\xffI\xffk\xff\x85\xff\xa9\xff\xc4\xff\xdd\xff\xf5\xff\x15\x00<\x00\\\x00\x81\x00\xa9\x00\xd0\x00\xf6\x00\x16\x013\x01F\x01O\x01P\x01N\x01H\x016\x01)\x01\x14\x01\xf5\x00\xd2\x00\xa9\x00w\x00Q\x001\x00\x08\x00\xdc\xff\xa8\xffp\xff<\xff\x0c\xff\xdf\xfe\xba\xfe\xa3\xfe\x8b\xfev\xfer\xfep\xfe{\xfe\x9f\xfe\xba\xfe\xcd\xfe\xe5\xfe\xf7\xfe\x0e\xff:\xffr\xff\xb2\xff\xfb\xff7\x00n\x00\x99\x00\xc5\x00\xe6\x00\xfa\x00\x0e\x01\x18\x01\x1f\x01\'\x012\x01C\x01\\\x01m\x01f\x01G\x01\x1a\x01\xda\x00\x9c\x00d\x001\x00\x06\x00\xe4\xff\xbe\xff\x92\xfff\xff;\xff\x16\xff\xf1\xfe\xda\xfe\xb3\xfe\x97\xfe~\xfee\xfeZ\xfeY\xfer\xfe\x80\xfe\xa4\xfe\xd1\xfe\xfb\xfe-\xffO\xffo\xff\x84\xff\xb1\xff\xd7\xff\xfb\xff9\x00j\x00\x9c\x00\xd3\x00\xfc\x00\x1e\x01B\x01N\x01N\x01K\x01:\x01\x1d\x01\x03\x01\xef\x00\xe0\x00\xda\x00\xde\x00\xc8\x00\x9c\x00f\x00,\x00\x00\x00\xe6\xff\xe0\xff\xdb\xff\xd0\xff\xc9\xff\xad\xff\x88\xffl\xffD\xff\x11\xff\xd5\xfe\x9c\xfer\xfeU\xfeU\xfe`\xfe\x81\xfe\xb2\xfe\xd9\xfe\x00\xff\'\xffZ\xff\x84\xff\xa9\xff\xcc\xff\xe4\xff\x03\x00\'\x00S\x00\x83\x00\xbe\x00\xe3\x00\xf7\x00\x10\x01\x0e\x01\x06\x01\x08\x01\x00\x01\x01\x01\x04\x01\x03\x01\xfd\x00\xe9\x00\xd1\x00\xba\x00\x9d\x00\x80\x00Y\x00,\x00\r\x00\xe2\xff\xc2\xff\xab\xff\x9c\xff\x84\xffr\xffh\xffO\xff=\xff(\xff\t\xff\xf7\xfe\xf3\xfe\xe8\xfe\xe2\xfe\xe5\xfe\xf2\xfe\n\xff1\xffj\xff\xa5\xff\xe1\xff\x1d\x00Q\x00\x8c\x00\xc7\x00\xf5\x00"\x01A\x01O\x01V\x01R\x01I\x01C\x01#\x01\x0c\x01\xf4\x00\xc8\x00\xab\x00\x8b\x00\x83\x00\x7f\x00}\x00\x90\x00\x94\x00\x9b\x00\xa8\x00\xa6\x00\x9f\x00\x93\x00u\x00R\x00\'\x00\x00\x00\xd3\xff\xac\xff\x8e\xff_\xffB\xff4\xff*\xff\x1e\xff\x0f\xff\r\xff\x14\xff%\xffM\xff\x85\xff\xc5\xff\x16\x00W\x00\x8f\x00\xba\x00\xd4\x00\xe3\x00\xdc\x00\xd9\x00\xbf\x00\xa7\x00\x96\x00\x88\x00\x82\x00|\x00w\x00o\x00g\x00\\\x00U\x00:\x00%\x00\x0c\x00\xf3\xff\xf3\xff\xe9\xff\xde\xff\xc3\xff\x96\xffj\xff)\xff\xe5\xfe\xb0\xfe}\xfe\\\xfeM\xfeT\xfe\x7f\xfe\xb0\xfe\xf2\xfe.\xff]\xff\x9c\xff\xc4\xff\xef\xff\x1c\x00E\x00w\x00\xb4\x00\xf2\x00*\x01V\x01z\x01\x89\x01|\x01m\x01K\x01\x1a\x01\xf4\x00\xc9\x00\xb2\x00\x9e\x00\x80\x00s\x00W\x00B\x00+\x00\x10\x00\xf9\xff\xda\xff\xb5\xff\x8b\xffc\xff6\xff\x1b\xff\t\xff\xfc\xfe\xf3\xfe\xdc\xfe\xd1\xfe\xc6\xfe\xb6\xfe\xac\xfe\xb1\xfe\xc4\xfe\xdb\xfe\xff\xfe/\xffp\xff\xab\xff\xe8\xff\x1e\x00Q\x00|\x00\x93\x00\xb1\x00\xc5\x00\xdc\x00\xe9\x00\xf4\x00\t\x01\x15\x01\x1e\x01\'\x010\x01)\x01\x12\x01\xf7\x00\xd4\x00\x9f\x00p\x00=\x00\x11\x00\xeb\xff\xc0\xff\xae\xff\x8a\xffp\xffW\xff:\xff\'\xff\x13\xff\x08\xff\xf6\xfe\xdc\xfe\xc5\xfe\xbb\xfe\xb3\xfe\xc2\xfe\xe8\xfe\x02\xff\x16\xff(\xff3\xffB\xffg\xff\x94\xff\xcc\xff\x04\x000\x00b\x00\x8d\x00\xc1\x00\xf7\x00\x1d\x018\x01?\x01?\x01;\x01-\x014\x018\x012\x018\x01=\x019\x01\x19\x01\xf5\x00\xc2\x00z\x008\x00\xf9\xff\xb8\xff\x80\xffR\xff*\xff\x0b\xff\xf0\xfe\xc6\xfe\x92\xfec\xfe?\xfe&\xfe\x1f\xfe!\xfe\x1e\xfe9\xfeY\xfe\x89\xfe\xcc\xfe\x12\xffa\xff\x9b\xff\xcd\xff\xf6\xff\x1a\x00H\x00~\x00\xad\x00\xe0\x00\x10\x014\x01U\x01l\x01\x83\x01\x98\x01\xa0\x01\x9c\x01\x97\x01\x99\x01\x8e\x01\x80\x01l\x01I\x01.\x01\x04\x01\xd3\x00\x96\x00Z\x00\x08\x00\xb1\xffd\xff \xff\x00\xff\xf0\xfe\xed\xfe\xe3\xfe\xd4\xfe\xb5\xfe\x97\xfe\x83\xfe\x81\xfe\x8f\xfe\xa0\xfe\xbe\xfe\xd3\xfe\xe5\xfe\x01\xff*\xffP\xff|\xff\xa7\xff\xbf\xff\xe4\xff\x03\x00&\x00X\x00|\x00\xa6\x00\xd3\x00\xf8\x00\x1d\x01>\x01N\x01d\x01\x87\x01\x9e\x01\xb2\x01\xa4\x01\x90\x01r\x01J\x01+\x01\xff\x00\xd9\x00\xb2\x00\x8a\x00\\\x00)\x00\xfb\xff\xbc\xff\x7f\xffB\xff\x08\xff\xdc\xfe\xc1\xfe\xb4\xfe\xb1\xfe\xbb\xfe\xc3\xfe\xd3\xfe\xef\xfe\r\xff\x1f\xffA\xffi\xff\x91\xff\xcb\xff\x0b\x00F\x00\x7f\x00\xb8\x00\xe0\x00\xf7\x00\x0b\x01\xfc\x00\xdc\x00\xbf\x00\xb0\x00\xa5\x00\x93\x00\x92\x00\x8e\x00~\x00n\x00k\x00m\x00v\x00u\x00f\x00P\x007\x00\x16\x00\xf4\xff\xd9\xff\xb1\xff\x8d\xff_\xff2\xff\x08\xff\xdc\xfe\xc5\xfe\xbe\xfe\xc9\xfe\xec\xfe\x15\xff7\xffZ\xff{\xff\xae\xff\xea\xff\x0f\x009\x00J\x00T\x00W\x00P\x00[\x00S\x00R\x00h\x00\x83\x00\xa6\x00\xc9\x00\xe5\x00\xed\x00\xf0\x00\xf1\x00\xf3\x00\xfd\x00\xff\x00\xed\x00\xd3\x00\xac\x00y\x00]\x004\x00\xfd\xff\xc5\xff\x88\xffM\xff\x1e\xff\x08\xff\xf3\xfe\xe0\xfe\xd7\xfe\xda\xfe\xd6\xfe\xd9\xfe\xe6\xfe\xee\xfe\xf3\xfe\xef\xfe\xf5\xfe\x04\xff\x0f\xff(\xffF\xffl\xff\x95\xff\xc4\xff\xf2\xff\x11\x004\x00Q\x00r\x00\x93\x00\xbd\x00\xd2\x00\xdf\x00\xee\x00\xff\x00\x10\x01\x1f\x01.\x01\x18\x01\xff\x00\xd8\x00\xa8\x00\x81\x00^\x00>\x00\x15\x00\xf9\xff\xdc\xff\xb3\xff\x8e\xff[\xff+\xff\xfd\xfe\xda\xfe\xbc\xfe\xaa\xfe\x9e\xfe\x99\xfe\xa3\xfe\xb3\xfe\xda\xfe\xf3\xfe\x16\xff9\xffL\xff]\xffh\xff\x7f\xff\x9d\xff\xbf\xff\xd9\xff\x01\x00-\x00Z\x00\x8d\x00\xc1\x00\xed\x00\x06\x01+\x018\x019\x01R\x01m\x01v\x01o\x01a\x01?\x01!\x01\xfd\x00\xd3\x00\xa7\x00c\x00\x1a\x00\xb6\xffl\xff:\xff\r\xff\xf7\xfe\xe3\xfe\xc5\xfe\xa4\xfe\x90\xfe\x7f\xfe\x8a\xfe\x94\xfe\x9d\xfe\xa6\xfe\xb8\xfe\xee\xfe\'\xffc\xff\x91\xff\xb6\xff\xd5\xff\xe1\xff\x05\x00\x1c\x000\x00=\x00X\x00x\x00\x8d\x00\xbe\x00\xee\x00\x1c\x013\x01J\x01`\x01q\x01o\x01m\x01t\x01^\x01W\x01N\x01<\x01\x17\x01\xe8\x00\xb3\x00y\x00L\x00\x14\x00\xda\xff\x9d\xff]\xff-\xff\t\xff\xf0\xfe\xd5\xfe\xcb\xfe\xbc\xfe\xaf\xfe\xaf\xfe\xab\xfe\xb5\xfe\xc8\xfe\xe1\xfe\xf9\xfe\x1c\xffE\xffq\xff\x98\xff\xb0\xff\xde\xff\x0b\x00=\x00_\x00}\x00\x9e\x00\xb3\x00\xe3\x00\x1d\x01H\x01k\x01\x89\x01\x91\x01\x8c\x01\x94\x01\x8c\x01y\x01_\x01?\x01\x17\x01\xee\x00\xc8\x00\x9b\x00p\x003\x00\xfb\xff\xc0\xff\x8d\xff\\\xff!\xff\xf9\xfe\xd8\xfe\xc2\xfe\xad\xfe\xaf\xfe\xad\xfe\xb9\xfe\xd2\xfe\xda\xfe\xe7\xfe\xe7\xfe\xf1\xfe\x02\xff\x11\xff8\xffb\xff\x8a\xff\xb0\xff\xe3\xff\x14\x009\x00]\x00z\x00\x95\x00\xad\x00\xcf\x00\xf0\x00\t\x01\x1f\x01/\x01@\x01?\x01<\x01.\x01\r\x01\xe0\x00\xb1\x00\x84\x00Q\x001\x00\x19\x00\x01\x00\xe7\xff\xd0\xff\xb3\xff\xac\xff\x8e\xfft\xff[\xff2\xff\x13\xff\xff\xfe\xfa\xfe\xfb\xfe\xfd\xfe\x19\xff2\xff3\xffD\xffO\xffW\xffp\xff\x91\xff\xbb\xff\xe7\xff\x1c\x00F\x00b\x00|\x00\x95\x00\xba\x00\xd1\x00\xe0\x00\xe9\x00\xe0\x00\xd3\x00\xd0\x00\xd6\x00\xce\x00\xd0\x00\xce\x00\xc0\x00\xb1\x00\xa2\x00\x8d\x00g\x00:\x00\x0f\x00\xed\xff\xda\xff\xc2\xff\xa8\xff\x81\xffM\xff\'\xff\t\xff\xfe\xfe\xf2\xfe\xf7\xfe\xee\xfe\xd6\xfe\xdd\xfe\xe0\xfe\xf2\xfe\x0e\xff\'\xffE\xffi\xff\x94\xff\xc8\xff\xf7\xff\x19\x00B\x00o\x00\x95\x00\xc0\x00\xdd\x00\xfc\x00\x12\x01%\x019\x01<\x018\x013\x01\x14\x01\xf5\x00\xde\x00\xbd\x00\x99\x00m\x00?\x00\x12\x00\xf3\xff\xdb\xff\xbd\xff\xa5\xffx\xffT\xff9\xff\x1c\xff\x10\xff\r\xff\x0b\xff\xff\xfe\x04\xff\x0c\xff\x13\xff$\xff<\xffN\xff`\xff}\xff\x93\xff\xb7\xff\xd9\xff\xf8\xff"\x00@\x00]\x00u\x00\x8f\x00\xa4\x00\xbc\x00\xe0\x00\xed\x00\x07\x01\x10\x01\xf9\x00\xe4\x00\xde\x00\xd2\x00\xc4\x00\xc5\x00\xba\x00\xaa\x00\x83\x00e\x00>\x00\x11\x00\xf4\xff\xc4\xff\xa3\xff}\xff`\xff=\xff\x1a\xff\x00\xff\xd0\xfe\xb5\xfe\x9e\xfe\x99\xfe\x9e\xfe\xa0\xfe\xbf\xfe\xcf\xfe\xf4\xfe"\xffG\xffm\xff\x8c\xff\xa8\xff\xbe\xff\xdb\xff\xf5\xff\x15\x00=\x00i\x00\xa1\x00\xd5\x00\x01\x01!\x017\x01[\x01u\x01\xa8\x01\xdb\x01\xe7\x01\xd4\x01\xb0\x01z\x01G\x01\x0c\x01\xc8\x00~\x00(\x00\xda\xff\x99\xffk\xffE\xff\'\xff\x04\xff\xe0\xfe\xca\xfe\xbf\xfe\xca\xfe\xcb\xfe\xd0\xfe\xcb\xfe\xc2\xfe\xbf\xfe\xc5\xfe\xdf\xfe\xec\xfe\xfe\xfe\x16\xff*\xffK\xffx\xff\xae\xff\xe5\xff\x1f\x00e\x00\x99\x00\xcb\x00\xf2\x00\x1c\x01B\x01`\x01{\x01\x95\x01\xa8\x01\xb1\x01\xab\x01\xab\x01\xaa\x01\xa9\x01\xa2\x01\x94\x01\x81\x01H\x01\x10\x01\xce\x00\x84\x00-\x00\xdf\xff\x8b\xff,\xff\xdf\xfe\x89\xfeA\xfe\x0f\xfe\xec\xfd\xdc\xfd\xed\xfd\x0f\xfe,\xfeY\xfe\x8c\xfe\xc9\xfe\x16\xffX\xff\x94\xff\xad\xff\xc8\xff\xdd\xff\xe3\xff\xf7\xff\x0c\x00,\x00L\x00r\x00\xa6\x00\xdb\x00\x10\x019\x01]\x01w\x01\x99\x01\xc7\x01\xe1\x01\xea\x01\xed\x01\xe4\x01\xc8\x01\x9c\x01j\x01)\x01\xe1\x00\x92\x00C\x00\xf6\xff\xb0\xff\x80\xff@\xff\x02\xff\xd7\xfe\xa2\xfe\x7f\xfed\xfeV\xfeR\xfeU\xfew\xfe\x95\xfe\xad\xfe\xc4\xfe\xd2\xfe\xdc\xfe\xe3\xfe\xfd\xfe\x1c\xff9\xffR\xffs\xff\x8c\xff\xb3\xff\xf0\xff$\x00a\x00\x9a\x00\xd2\x00\xff\x00/\x01d\x01s\x01\x83\x01\x8a\x01z\x01l\x01H\x01\x17\x01\xd7\x00\x8c\x00O\x00\x1b\x00\xf1\xff\xcd\xff\x9e\xffr\xffV\xff<\xff,\xff\x19\xff\x08\xff\xf9\xfe\xda\xfe\xda\xfe\xef\xfe\xff\xfe\x16\xff2\xffH\xff_\xff\x80\xff\x99\xff\xa3\xff\xb2\xff\xb4\xff\xc5\xff\xe3\xff\x07\x004\x00O\x00j\x00\x80\x00\xa2\x00\xbb\x00\xd4\x00\xe8\x00\xff\x00\x18\x01,\x01:\x012\x01"\x01\r\x01\xef\x00\xd9\x00\xc3\x00\x9a\x00f\x004\x00\x05\x00\xe1\xff\xcc\xff\xb7\xff\x98\xff|\xffq\xff`\xff\\\xffW\xffA\xff)\xff\x0f\xff\x04\xff\xf5\xfe\xeb\xfe\xfa\xfe\xf0\xfe\xe8\xfe\xf3\xfe\x10\xff7\xff[\xff\x8b\xff\xb1\xff\xd0\xff\x04\x00,\x00U\x00\x8a\x00\xa4\x00\xc4\x00\xed\x00\xff\x00\r\x01\x14\x01\x1a\x01\x18\x01\x1c\x01"\x01\x1a\x01\x16\x01\xfa\x00\xc7\x00\x97\x00b\x00-\x00\xff\xff\xda\xff\xac\xff|\xff_\xffJ\xff=\xff0\xff\x1f\xff\x15\xff\x04\xff\xf2\xfe\xe7\xfe\xdc\xfe\xdd\xfe\xe5\xfe\xe9\xfe\x02\xff \xff6\xffP\xffw\xff\xa8\xff\xd6\xff\x15\x00P\x00\x8a\x00\xcd\x00\xf6\x00\x1b\x01?\x01E\x017\x01+\x01$\x01\x12\x01\x04\x01\x05\x01\xe1\x00\xc5\x00\xab\x00\x88\x00m\x00N\x00$\x00\x04\x00\xde\xff\xbd\xff\x98\xffy\xff^\xffR\xff?\xff>\xffC\xff \xff\x05\xff\xe6\xfe\xdb\xfe\xe2\xfe\xe9\xfe\x07\xff\x1a\xff>\xff_\xff\x8e\xff\xce\xff\x05\x00B\x00u\x00\xb0\x00\xe6\x00\x1b\x01M\x01x\x01\xa3\x01\xbf\x01\xcc\x01\xd4\x01\xd6\x01\xbc\x01\x92\x01m\x01?\x01#\x01\x0c\x01\xf8\x00\xe1\x00\xad\x00b\x00\x17\x00\xdf\xff\xaa\xff\x87\xffr\xffN\xff$\xff\xfc\xfe\xe0\xfe\xb9\xfe\x9d\xfex\xfeU\xfeF\xfe<\xfe=\xfeG\xfeN\xfed\xfe\x87\xfe\xa6\xfe\xc6\xfe\xed\xfe\x18\xffC\xfft\xff\xb2\xff\xf7\xffB\x00\x8a\x00\xc9\x00\xeb\x00\x10\x01A\x01b\x01\x82\x01\x9c\x01\xb2\x01\xb6\x01\xb2\x01\xc0\x01\xb2\x01\x9c\x01t\x01@\x01\x10\x01\xde\x00\xa8\x00q\x00:\x00\x03\x00\xc9\xff\x92\xffd\xff0\xff\x03\xff\xe4\xfe\xc7\xfe\xad\xfe\x93\xfe\x8b\xfe\x86\xfe|\xfe\x81\xfe\x80\xfe\x8b\xfe\x94\xfe\x97\xfe\xa8\xfe\xab\xfe\xc7\xfe\xe7\xfe\x0b\xff?\xffc\xff\x94\xff\xc0\xff\x02\x00L\x00\x84\x00\xc4\x00\xf4\x00#\x01;\x01C\x01F\x019\x01#\x01\x0c\x01\xf4\x00\xdc\x00\xc2\x00\xa2\x00\x8f\x00\x7f\x00p\x00b\x00Z\x00P\x003\x00\x1e\x00\x11\x00\xf6\xff\xe1\xff\xbc\xff\x9c\xffx\xffE\xff&\xff\xfe\xfe\xd7\xfe\xce\xfe\xc1\xfe\xd4\xfe\xf6\xfe\x11\xff:\xff`\xff\x8e\xff\xa8\xff\xd6\xff\x15\x00D\x00|\x00\xa5\x00\xcc\x00\xec\x00\xfc\x00\x13\x01"\x01,\x01#\x01\x17\x01\t\x01\xef\x00\xd6\x00\xbb\x00\xa5\x00\x98\x00\x89\x00i\x00A\x00\r\x00\xd7\xff\xa2\xffm\xffL\xff \xff\x06\xff\xf1\xfe\xd7\xfe\xca\xfe\xb8\xfe\xad\xfe\xa9\xfe\x9d\xfe\x9b\xfe\x99\xfe\xac\xfe\xcc\xfe\xde\xfe\xfb\xfe\x1b\xffG\xff\x7f\xff\xbd\xff\x0c\x00?\x00c\x00\x82\x00\x9b\x00\xba\x00\xe7\x00\n\x01#\x017\x01<\x015\x01-\x01%\x01\x12\x01\xfc\x00\xec\x00\xe3\x00\xd3\x00\xb8\x00\x9b\x00w\x00\\\x00@\x00\x16\x00\xf9\xff\xdc\xff\xb8\xff\x8d\xffj\xff;\xff$\xff\x03\xff\xe1\xfe\xd4\xfe\xc2\xfe\xbc\xfe\xbe\xfe\xd7\xfe\xf3\xfe\x16\xffA\xffb\xff\x94\xff\xc9\xff\xf8\xff\x19\x004\x00V\x00f\x00\x84\x00\xa8\x00\xc9\x00\xe3\x00\xe7\x00\xdb\x00\xc7\x00\xc4\x00\xc8\x00\xc7\x00\xb0\x00\x9e\x00\x80\x00j\x00b\x00R\x00H\x00:\x00(\x00\x0c\x00\xef\xff\xde\xff\xce\xff\xb2\xff\x9d\xff\x8f\xff}\xffl\xff[\xffO\xff?\xff:\xffB\xff0\xff8\xffN\xffX\xff\x7f\xff\xa6\xff\xd2\xff\xf9\xff\'\x00N\x00u\x00\xaa\x00\xdb\x00\x0c\x01#\x01@\x01Z\x01]\x01e\x01f\x01d\x01`\x01U\x016\x01\x18\x01\xfb\x00\xda\x00\xc0\x00\xa4\x00\x86\x00W\x00,\x00\x03\x00\xc9\xff\x94\xff^\xff4\xff\x04\xff\xdc\xfe\xb4\xfe\x87\xfek\xfeW\xfeK\xfeN\xfeN\xfeI\xfeY\xfe|\xfe\xa7\xfe\xe1\xfe\x1c\xffT\xff\x8f\xff\xc4\xff\xf7\xff+\x00^\x00\x8e\x00\xbd\x00\xeb\x00\x0b\x01 \x010\x016\x01-\x01%\x01\x1e\x01\x19\x01\x11\x01\xfc\x00\xee\x00\xd7\x00\xb4\x00\x90\x00k\x00D\x00\x1c\x00\x00\x00\xde\xff\xc4\xff\xa6\xff\x85\xffc\xff;\xff"\xff\x07\xff\xee\xfe\xd8\xfe\xcd\xfe\xbc\xfe\xba\xfe\xc5\xfe\xcc\xfe\xdd\xfe\xeb\xfe\xfd\xfe\x10\xff)\xffD\xffc\xff\x8b\xff\xb0\xff\xe7\xff\x1e\x00J\x00s\x00\x94\x00\xb6\x00\xd8\x00\xfb\x00\x1a\x01=\x01b\x01|\x01\x90\x01\x98\x01\x88\x01\x85\x01{\x01R\x01/\x01\x05\x01\xd6\x00\xad\x00\x8d\x00^\x00+\x00\xfb\xff\xc2\xff\x8e\xff^\xff"\xff\xea\xfe\xc7\xfe\xa7\xfe\x93\xfe\x92\xfe\x96\xfe\x94\xfe\xa3\xfe\xa4\xfe\xa6\xfe\xb6\xfe\xd2\xfe\xf4\xfe\x1a\xffD\xffv\xff\xa5\xff\xd3\xff\x07\x00?\x00\x7f\x00\xb6\x00\xdf\x00\x00\x01\x19\x01\'\x018\x01R\x01h\x01r\x01w\x01c\x01J\x014\x01\x17\x01\xff\x00\xe8\x00\xd3\x00\xbb\x00\x95\x00g\x000\x00\xfd\xff\xd7\xff\xb5\xff\x93\xffh\xffI\xff-\xff\n\xff\xfb\xfe\xed\xfe\xde\xfe\xd4\xfe\xb9\xfe\xa5\xfe\xb0\xfe\xbf\xfe\xdb\xfe\xeb\xfe\x07\xff \xffE\xffx\xff\x99\xff\xc0\xff\xdb\xff\xe7\xff\x0e\x008\x00[\x00\x85\x00\xb8\x00\xdc\x00\xee\x00\x08\x01\x18\x01\x18\x01!\x01\x19\x01\x19\x01\x16\x01\x07\x01\xe4\x00\xce\x00\xb2\x00\x84\x00U\x002\x00\x02\x00\xd3\xff\xac\xff\x7f\xffg\xffR\xff=\xff4\xff\x1b\xff\xfd\xfe\xea\xfe\xe4\xfe\xf4\xfe\xf8\xfe\x04\xff\x14\xff \xff(\xff3\xffK\xffk\xff\x8a\xff\xb7\xff\xe0\xff\x06\x00<\x00R\x00q\x00\xa5\x00\xce\x00\xee\x00\x07\x01%\x01&\x01*\x013\x01\x1e\x01\x16\x01\x06\x01\xef\x00\xd8\x00\xb9\x00\xb1\x00\xa0\x00\x8b\x00g\x00:\x00\x16\x00\xf2\xff\xc2\xff\x9c\xffs\xffH\xff\x1b\xff\xf8\xfe\xde\xfe\xd8\xfe\xda\xfe\xd8\xfe\xee\xfe\x11\xff=\xffn\xff\xa2\xff\xd0\xff\x05\x009\x00c\x00{\x00\x8d\x00\x90\x00\x92\x00\x9a\x00\xa7\x00\xb4\x00\xb6\x00\xac\x00\xaf\x00\xb5\x00\xb8\x00\xc3\x00\xca\x00\xce\x00\xce\x00\xc6\x00\xb7\x00\x9e\x00|\x00I\x00\x10\x00\xe6\xff\xb8\xff\x8a\xfff\xff@\xff*\xff\x16\xff\x0b\xff\x05\xff\xf9\xfe\xfa\xfe\xfa\xfe\xfa\xfe\x10\xff \xff.\xff9\xffJ\xff[\xffx\xff\x99\xff\xb0\xff\xd6\xff\xf6\xff\x13\x00C\x00r\x00\x9d\x00\xb7\x00\xb9\x00\xc5\x00\xcf\x00\xe1\x00\xe6\x00\xde\x00\xd2\x00\xad\x00\x9a\x00\x82\x00n\x00S\x003\x00\x1b\x00\x10\x00\x10\x00\x12\x00\x0e\x00\x08\x00\x00\x00\xed\xff\xde\xff\xc7\xff\xa8\xffz\xff=\xff\x14\xff\xf6\xfe\xce\xfe\xbd\xfe\xbc\xfe\xb7\xfe\xc1\xfe\xde\xfe\n\xff8\xffo\xff\x9b\xff\xcf\xff\x18\x00c\x00\xb1\x00\xf5\x007\x01f\x01\x87\x01\xa0\x01\x9d\x01\x8e\x01\x8e\x01t\x01Y\x01G\x01\x1b\x01\xf7\x00\xca\x00\x9b\x00\x87\x00n\x00K\x00.\x00 \x00\x12\x00\x06\x00\xfa\xff\xe5\xff\xc7\xff\xa2\xff\x83\xff`\xff,\xff\xf9\xfe\xbb\xfe{\xfeK\xfe \xfe\x07\xfe\x08\xfe\x1d\xfe6\xfeP\xfe\x80\xfe\xc0\xfe\xf8\xfe?\xff\x88\xff\xc5\xff\xfc\xff%\x00F\x00]\x00f\x00c\x00f\x00\x83\x00\x9a\x00\xb8\x00\xce\x00\xde\x00\xf4\x00\x06\x01\x16\x01\x1c\x01\x1e\x01\x1e\x01\x12\x01\x08\x01\xf7\x00\xcf\x00\xb5\x00\x97\x00z\x00l\x00N\x00 \x00\xf0\xff\xbd\xff\x86\xffP\xff)\xff\x10\xff\xf6\xfe\xe5\xfe\xd5\xfe\xcc\xfe\xb5\xfe\xb4\xfe\xbd\xfe\xc1\xfe\xea\xfe\x10\xff4\xff\\\xff\x87\xff\xa6\xff\xd1\xff\n\x00<\x00y\x00\xb3\x00\xd5\x00\xf4\x00!\x01:\x01L\x01b\x01b\x01n\x01o\x01i\x01]\x01D\x01/\x01\x07\x01\xe3\x00\xcb\x00\x9f\x00v\x006\x00\xf5\xff\xc7\xff\x8e\xff[\xff3\xff\x0c\xff\xeb\xfe\xd0\xfe\xbe\xfe\xa9\xfe\x9f\xfe\x98\xfe\x91\xfe\xa0\xfe\xaa\xfe\xc5\xfe\xf3\xfe\x0f\xff6\xffX\xff\x80\xff\xbc\xff\xf2\xff,\x00h\x00\x95\x00\xc5\x00\xf3\x00*\x01R\x01g\x01\x83\x01\x8b\x01\x89\x01\x81\x01d\x01;\x01$\x01\x15\x01\xfa\x00\xe0\x00\xa9\x00k\x00>\x00)\x00\x13\x00\x00\x00\xe4\xff\xa9\xffm\xff>\xff\x1f\xff\x19\xff\r\xff\x07\xff\x05\xff\xfe\xfe\xfa\xfe\x00\xff\xff\xfe\x06\xff\x1b\xff\x1d\xff/\xffM\xffa\xffw\xff\x8f\xff\xa9\xff\xc5\xff\xdc\xff\xef\xff\x06\x00%\x00A\x00h\x00\x87\x00\x97\x00\x9c\x00\x98\x00\x97\x00\xa7\x00\xc8\x00\xe5\x00\xf1\x00\xf6\x00\xe3\x00\xca\x00\xbd\x00\xa6\x00\x89\x00y\x00@\x00\x00\x00\xe4\xff\xc2\xff\x9e\xff\x92\xff}\xffg\xffk\xffk\xffi\xfff\xff\\\xffM\xffF\xffa\xff\x80\xff\x9a\xff\xb6\xff\xc5\xff\xbb\xff\xb4\xff\xb0\xff\xad\xff\xc7\xff\xf1\xff\x19\x00<\x00X\x00`\x00h\x00u\x00}\x00\x95\x00\xaf\x00\xc7\x00\xc8\x00\xae\x00\x9e\x00\x84\x00g\x00W\x006\x00\r\x00\xf1\xff\xcb\xff\xad\xff\xa4\xff\x92\xff\x7f\xffy\xffi\xffY\xffT\xffJ\xffD\xffN\xff[\xffe\xffs\xff{\xff\x89\xff\x97\xff\xa5\xff\xb1\xff\xbb\xff\xd0\xff\xe6\xff\xfc\xff\x12\x009\x00k\x00\x98\x00\xb5\x00\xce\x00\xde\x00\xf2\x00\x04\x01\x06\x01\x00\x01\x02\x01\x06\x01\xfd\x00\x03\x01\xf8\x00\xde\x00\xc7\x00\xa5\x00\x90\x00\x83\x00u\x00`\x00A\x00\x1c\x00\xe5\xff\xa7\xffa\xff \xff\xea\xfe\xb2\xfe~\xfef\xfeH\xfe(\xfe/\xfe>\xfeV\xfe\x92\xfe\xcb\xfe\xfa\xfe8\xffg\xff\x83\xff\xa7\xff\xd0\xff\xeb\xff\x03\x00\x1b\x00%\x00!\x00/\x00R\x00t\x00\xad\x00\xe9\x00\x1e\x01O\x01\x81\x01\xa7\x01\xbe\x01\xda\x01\xf0\x01\xf8\x01\xe8\x01\xb2\x01f\x01\x0f\x01\xb6\x00X\x00\xfb\xff\xb3\xffs\xff@\xff \xff\x06\xff\xfb\xfe\xf6\xfe\xf3\xfe\xf7\xfe\xff\xfe\xf9\xfe\xf2\xfe\xe7\xfe\xd8\xfe\xcd\xfe\xcd\xfe\xd4\xfe\xda\xfe\xed\xfe\n\xff\x1c\xff,\xffZ\xff\x87\xff\xb5\xff\xff\xffC\x00o\x00\xac\x00\xe2\x00\xf1\x00\x11\x014\x01C\x01N\x01^\x01X\x01L\x01T\x01A\x01$\x01\x18\x01\xff\x00\xd3\x00\xa9\x00|\x00G\x00\r\x00\xea\xff\xb6\xff\x84\xff`\xff4\xff\x1c\xff\x17\xff\x19\xff\x1a\xff\x16\xff\x0c\xff\xfc\xfe\xe9\xfe\xf1\xfe\xfd\xfe\r\xff+\xff2\xff4\xff9\xff:\xffG\xffi\xff\x99\xff\xd1\xff\x10\x00M\x00s\x00\xa4\x00\xdf\x00\x11\x018\x01g\x01\x80\x01\x89\x01\x8d\x01x\x01[\x01=\x01\x14\x01\xe4\x00\xc9\x00\xb0\x00\x8f\x00\x82\x00s\x00Q\x00,\x00\x08\x00\xee\xff\xd2\xff\xc8\xff\xb3\xff~\xffK\xff\x17\xff\xe7\xfe\xcd\xfe\xcc\xfe\xcd\xfe\xd6\xfe\xeb\xfe\xf7\xfe\xf6\xfe\x01\xff"\xff;\xffa\xff\x9e\xff\xcb\xff\xf1\xff!\x00@\x00r\x00\xa8\x00\xc7\x00\xeb\x00\xf9\x00\x05\x01\x1a\x01.\x01:\x01@\x01B\x012\x01!\x01\x0f\x01\xed\x00\xce\x00\xad\x00s\x00/\x00\xf0\xff\xb7\xff\x93\xffu\xffa\xffT\xffI\xffA\xff0\xff4\xff1\xff\x1d\xff\x14\xff\x1f\xff\x15\xff\x15\xff\x10\xff\xfe\xfe\r\xff\x15\xff$\xff1\xff:\xffP\xffl\xff\x9a\xff\xc8\xff\xee\xff\x0c\x00-\x00Z\x00y\x00\x9c\x00\xc3\x00\xd0\x00\xe0\x00\xec\x00\xe5\x00\xd2\x00\xb5\x00\x95\x00~\x00m\x00c\x00U\x00;\x00%\x00\x07\x00\xe7\xff\xce\xff\xbf\xff\xb6\xff\x9b\xffz\xff\\\xffA\xff4\xff4\xff0\xff#\xff/\xffA\xffH\xffV\xffe\xffm\xffx\xff\x8e\xff\xb0\xff\xd3\xff\xf1\xff\x0c\x00&\x00A\x00`\x00u\x00\x94\x00\xba\x00\xde\x00\xfc\x00\r\x01\x19\x01\x0e\x01\xfd\x00\xe7\x00\xce\x00\xbd\x00\xaa\x00\x8c\x00p\x00P\x00+\x00\n\x00\xeb\xff\xcb\xff\xad\xff\x97\xff\x80\xff\\\xffM\xff9\xff\x17\xff\x05\xff\xef\xfe\xd4\xfe\xd7\xfe\xd5\xfe\xda\xfe\xe8\xfe\x05\xff(\xffC\xff{\xff\xaf\xff\xe2\xff\x1b\x00J\x00\x80\x00\xaf\x00\xcf\x00\xeb\x00\x04\x01\x19\x010\x01S\x01c\x01p\x01y\x01i\x01[\x01F\x011\x01 \x01\x06\x01\xeb\x00\xd1\x00\xaf\x00v\x007\x00\xfe\xff\xc0\xff\x88\xffV\xff*\xff\x07\xff\xf1\xfe\xdf\xfe\xd0\xfe\xcd\xfe\xcf\xfe\xe0\xfe\xef\xfe\xfa\xfe\x15\xff)\xff1\xff3\xffM\xffb\xff\x87\xff\xb9\xff\xde\xff\x0f\x00;\x00X\x00y\x00\xaa\x00\xd2\x00\xf3\x00+\x01P\x01^\x01z\x01\x81\x01u\x01w\x01g\x01K\x013\x01\x1a\x01\xff\x00\xeb\x00\xc4\x00\x98\x00s\x00>\x00\x05\x00\xce\xff\x98\xffe\xff9\xff\x18\xff\xed\xfe\xc0\xfe\x9c\xfe\x84\xfeo\xfeY\xfeO\xfeH\xfe@\xfeW\xfe\x88\xfe\xb2\xfe\xef\xfe+\xffX\xff\x89\xff\xbc\xff\xe7\xff\x10\x00D\x00~\x00\xaf\x00\xe4\x00\x17\x01A\x01e\x01x\x01u\x01x\x01v\x01c\x01W\x01?\x01%\x01\r\x01\xea\x00\xbb\x00\x83\x00H\x00\x02\x00\xce\xff\xa9\xffs\xffL\xff\x12\xff'} message_type=websocket.receive session_id=v5_mph3j0z2

(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> uv run uvicorn src.voice_commerce.main:app --reload
INFO:     Will watch for changes in these directories: ['P:\\AI_Empire\\Projects\\voice-commerce-agent']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [5316] using WatchFiles
INFO:     Started server process [11064]
INFO:     Waiting for application startup.
2026-03-24 06:28:59 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     127.0.0.1:4568 - "GET /static/test_client.html HTTP/1.1" 200 OK
2026-03-24 06:29:00 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:1969 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:29:00 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=1969)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:29:00 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:29:00 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:29:01 [info     ] gemini_session_connected      
2026-03-24 06:29:03 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [info     ] input_mode_switched_to_audio   session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:03 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:03 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:04 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:04 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:05 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:05 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:06 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:06 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:06 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:06 [debug    ] gemini_input_transcript        text=' <noise> ഹായ് <noise>'
2026-03-24 06:29:06 [info     ] user_speech_transcribed        session_id=v5_wtjkhgz9 text=' <noise> ഹായ് <noise>'
Warning: there are non-data parts in the response: ['text', 'thought'], returning concatenated data result from data parts, check out the non data parts for full response from model.
2026-03-24 06:29:07 [debug    ] gemini_output_transcript       text='Hello! How'
2026-03-24 06:29:07 [debug    ] gemini_output_transcript       text=' can I'
2026-03-24 06:29:08 [debug    ] gemini_output_transcript       text=' help'
Warning: there are non-text parts in the response: ['inline_data'], returning concatenated text result from text parts, check out the non text parts for full response from model.
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:29:08 [debug    ] gemini_output_transcript       text=' you today?'
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
INFO:     127.0.0.1:14239 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:08 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:29:09 [debug    ] turn_saved                     length=32 session=v5_wtjkhgz9
2026-03-24 06:29:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:20 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:20 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:20 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:20 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:23 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:23 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:29:23 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_wtjkhgz9
2026-03-24 06:29:23 [debug    ] gemini_sending_audio_chunk     bytes=8192
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-24 06:34:07 [info     ] gemini_session_closing        
2026-03-24 06:34:07 [info     ] gemini_session_closed_by_server reason='1000 None. '
2026-03-24 06:34:07 [info     ] gemini_session_closed         
2026-03-24 06:34:07 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=2
2026-03-24 06:34:07 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=2
INFO:     Waiting for application shutdown.
2026-03-24 06:34:07 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [11064]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Started server process [1544]
INFO:     Waiting for application startup.
2026-03-24 06:34:11 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-24 06:34:11 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [1544]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\__init__.py", line 2, in <module>
    from uvicorn.main import Server, main, run
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\main.py", line 32, in <module>
    from uvicorn.supervisors import ChangeReload, Multiprocess
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\__init__.py", line 12, in <module>
    from uvicorn.supervisors.watchfilesreload import WatchFilesReload as ChangeReload
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\watchfilesreload.py", line 7, in <module>
    from watchfiles import watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\__init__.py", line 2, in <module>
    from .main import Change, awatch, watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\main.py", line 9, in <module>
    import anyio
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\__init__.py", line 52, in <module>
    from ._core._streams import create_memory_object_stream as create_memory_object_stream
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_streams.py", line 7, in <module>
    from ..streams.memory import (
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1091, in get_code
  File "<frozen importlib._bootstrap_external>", line 1190, in get_data
KeyboardInterrupt
INFO:     Started server process [14720]
INFO:     Waiting for application startup.
2026-03-24 06:34:14 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:34:14 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:6550 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:34:14 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=6550)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:34:14 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:34:14 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:34:15 [info     ] gemini_session_connected      
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-24 06:34:17 [info     ] gemini_session_closing        
2026-03-24 06:34:17 [info     ] gemini_session_closed_by_server reason='1000 None. '
2026-03-24 06:34:17 [info     ] gemini_session_closed_gracefully reason='1000 None. ' session=v5_wtjkhgz9
2026-03-24 06:34:17 [info     ] gemini_session_closed         
2026-03-24 06:34:17 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=0
2026-03-24 06:34:17 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=0
INFO:     Waiting for application shutdown.
2026-03-24 06:34:17 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [14720]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\__init__.py", line 2, in <module>
    from uvicorn.main import Server, main, run
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\main.py", line 32, in <module>
    from uvicorn.supervisors import ChangeReload, Multiprocess
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\__init__.py", line 12, in <module>
    from uvicorn.supervisors.watchfilesreload import WatchFilesReload as ChangeReload
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\watchfilesreload.py", line 7, in <module>
    from watchfiles import watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\__init__.py", line 2, in <module>
    from .main import Change, awatch, watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\main.py", line 9, in <module>
    import anyio
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\__init__.py", line 32, in <module>
    from ._core._sockets import TCPConnectable as TCPConnectable
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1091, in get_code
  File "<frozen importlib._bootstrap_external>", line 1190, in get_data
KeyboardInterrupt
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Started server process [18616]
INFO:     Waiting for application startup.
2026-03-24 06:34:21 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-24 06:34:21 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [18616]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mahah\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\multiprocessing\spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\__init__.py", line 2, in <module>
    from uvicorn.main import Server, main, run
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\main.py", line 32, in <module>
    from uvicorn.supervisors import ChangeReload, Multiprocess
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\__init__.py", line 12, in <module>
    from uvicorn.supervisors.watchfilesreload import WatchFilesReload as ChangeReload
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\uvicorn\supervisors\watchfilesreload.py", line 7, in <module>
    from watchfiles import watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\__init__.py", line 2, in <module>
    from .main import Change, awatch, watch
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\watchfiles\main.py", line 9, in <module>
    import anyio
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\__init__.py", line 26, in <module>
    from ._core._fileio import AsyncFile as AsyncFile
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_fileio.py", line 27, in <module>
    from .. import to_thread
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\to_thread.py", line 14, in <module>
    from .abc import CapacityLimiter
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\abc\__init__.py", line 43, in <module>
    from .._core._synchronization import (
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_synchronization.py", line 13, in <module>
    from ._tasks import CancelScope
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_tasks.py", line 12, in <module>
    class _IgnoredTaskStatus(TaskStatus[object]):
  File "P:\AI_Empire\Projects\voice-commerce-agent\.venv\Lib\site-packages\anyio\_core\_tasks.py", line 12, in _IgnoredTaskStatus
    class _IgnoredTaskStatus(TaskStatus[object]):

KeyboardInterrupt
INFO:     Started server process [7424]
INFO:     Waiting for application startup.
2026-03-24 06:34:24 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:34:24 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:13768 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:34:24 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=13768)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:34:24 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:34:24 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:34:25 [info     ] gemini_session_connected      
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Shutting down
INFO:     connection closed
2026-03-24 06:35:36 [info     ] gemini_session_closing        
2026-03-24 06:35:36 [info     ] gemini_session_closed_by_server reason='1000 None. '
2026-03-24 06:35:36 [info     ] gemini_session_closed_gracefully reason='1000 None. ' session=v5_wtjkhgz9
2026-03-24 06:35:36 [info     ] gemini_session_closed         
2026-03-24 06:35:36 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=0
2026-03-24 06:35:36 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=0
INFO:     Waiting for application shutdown.
2026-03-24 06:35:36 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [7424]
WARNING:  WatchFiles detected changes in 'src\voice_commerce\handlers\voice_websocket_handler.py'. Reloading...
 INFO:     Started server process [19220]
INFO:     Waiting for application startup.
2026-03-24 06:35:39 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-03-24 06:35:39 [info     ] Application shutting down     
INFO:     Application shutdown complete.
INFO:     Finished server process [19220]
INFO:     Started server process [7820]
INFO:     Waiting for application startup.
2026-03-24 06:35:42 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:35:42 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:9405 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:35:42 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=9405)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:35:42 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:35:42 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:35:42 [info     ] gemini_session_connected      


(voice-commerce-agent) PS P:\AI_Empire\Projects\voice-commerce-agent> uv run uvicorn src.voice_commerce.main:app --reload
INFO:     Will watch for changes in these directories: ['P:\\AI_Empire\\Projects\\voice-commerce-agent']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [8608] using WatchFiles
INFO:     Started server process [14120]
INFO:     Waiting for application startup.
2026-03-24 06:55:06 [info     ] voice_commerce_agent_starting  debug=True model=gemini-2.5-flash-native-audio-preview-12-2025 port=8000
INFO:     Application startup complete.
2026-03-24 06:55:06 [info     ] voice_handler_created          session_id=v5_wtjkhgz9
INFO:     127.0.0.1:10217 - "WebSocket /ws/voice?session_id=v5_wtjkhgz9" [accepted]
2026-03-24 06:55:06 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=10217)" session_id=v5_wtjkhgz9
INFO:     connection open
2026-03-24 06:55:06 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:55:06 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
INFO:     127.0.0.1:1358 - "GET /static/test_client.html HTTP/1.1" 304 Not Modified
INFO:     connection closed
2026-03-24 06:55:07 [info     ] voice_handler_created          session_id=v5_mph3j0z2
INFO:     127.0.0.1:9769 - "WebSocket /ws/voice?session_id=v5_mph3j0z2" [accepted]
2026-03-24 06:55:07 [info     ] ws_connected                   client="Address(host='127.0.0.1', port=9769)" session_id=v5_mph3j0z2
INFO:     connection open
2026-03-24 06:55:07 [debug    ] gemini_live_handler_created    model=gemini-2.5-flash-native-audio-preview-12-2025 voice=Charon
2026-03-24 06:55:07 [info     ] gemini_session_connecting      model=gemini-2.5-flash-native-audio-preview-12-2025
2026-03-24 06:55:07 [info     ] gemini_session_connected      
2026-03-24 06:55:07 [info     ] gemini_session_closing        
2026-03-24 06:55:07 [info     ] gemini_session_closed         
2026-03-24 06:55:07 [info     ] ws_disconnected_normally       session_id=v5_wtjkhgz9 turns=0
2026-03-24 06:55:07 [info     ] ws_handler_finished            session_id=v5_wtjkhgz9 total_turns=0
2026-03-24 06:55:07 [info     ] gemini_session_connected      
2026-03-24 06:55:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:16 [info     ] input_mode_switched_to_audio   session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:17 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:17 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:18 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:18 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:18 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:21 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:21 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:21 [debug    ] gemini_input_transcript        text=' Hi, how are you?'
2026-03-24 06:55:21 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' Hi, how are you?'
2026-03-24 06:55:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:22 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:22 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:22 [debug    ] gemini_output_transcript       text="Hello! I'm"
2026-03-24 06:55:22 [debug    ] gemini_output_transcript       text=' good,'
2026-03-24 06:55:22 [debug    ] gemini_output_transcript       text=' thank'
Warning: there are non-text parts in the response: ['inline_data'], returning concatenated text result from text parts, check out the non text parts for full response from model.
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' you.'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
INFO:     127.0.0.1:5328 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' How can'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' I help'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_output_transcript       text=' you today?'
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:23 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:25 [debug    ] turn_saved                     length=53 session=v5_mph3j0z2
2026-03-24 06:55:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:32 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:33 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:33 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:34 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:34 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:35 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:35 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:35 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:35 [debug    ] gemini_input_transcript        text=' find what about you.'
2026-03-24 06:55:35 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' find what about you.'
Warning: there are non-data parts in the response: ['text', 'thought'], returning concatenated data result from data parts, check out the non data parts for full response from model.
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text="I'm here"
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' to help'
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' you find'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' everything'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' you need!'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_output_transcript       text=' What are'
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:37 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_output_transcript       text=' you looking'
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_output_transcript       text=' for?'
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:38 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:55:40 [debug    ] turn_saved                     length=72 session=v5_mph3j0z2
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:42 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:42 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:43 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:43 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:47 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:47 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:48 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:48 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:55:48 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:55:48 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:36 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:36 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:36 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:36 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:37 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:37 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:38 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:38 [debug    ] gemini_input_transcript        text=' <noise> Oi, oi.'
2026-03-24 06:58:38 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' <noise> Oi, oi.'
2026-03-24 06:58:38 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:38 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:38 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:39 [debug    ] gemini_output_transcript       text="I'm actually"
2026-03-24 06:58:39 [debug    ] gemini_output_transcript       text=' a voice'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=46080
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' shopping'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' assistant.'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' How can'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' I help'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' you shop'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_output_transcript       text=' today?'
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:40 [debug    ] gemini_audio_chunk             bytes=1920
2026-03-24 06:58:43 [debug    ] turn_saved                     length=71 session=v5_mph3j0z2
2026-03-24 06:58:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 06:58:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 06:58:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:08 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:08 [debug    ] text_message_parsed_as_json    text='What are today'
2026-03-24 07:01:08 [info     ] text_message_received          preview='What are today' session_id=v5_mph3j0z2
2026-03-24 07:01:08 [debug    ] gemini_sending_text            text='What are today'
2026-03-24 07:01:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:44 [info     ] input_mode_switched_to_audio   session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:44 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:44 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:45 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:45 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:46 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:46 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:49 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:49 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' So'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' So'
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' are'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' are'
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' you'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' you'
2026-03-24 07:01:50 [debug    ] gemini_input_transcript        text=' li'
2026-03-24 07:01:50 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' li'
2026-03-24 07:01:50 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:50 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:51 [debug    ] gemini_input_transcript        text=stening
2026-03-24 07:01:51 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=stening
2026-03-24 07:01:51 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:51 [debug    ] gemini_input_transcript        text=?
2026-03-24 07:01:51 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=?
2026-03-24 07:01:51 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:01:51 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:01:51 [debug    ] gemini_input_transcript        text=' So'
2026-03-24 07:01:51 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' So'
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:11 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:11 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:12 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:12 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:13 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:13 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [debug    ] gemini_input_transcript        text=' Ste'
2026-03-24 07:02:14 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' Ste'
2026-03-24 07:02:14 [debug    ] gemini_input_transcript        text=el
2026-03-24 07:02:14 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=el
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:14 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:14 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=' '
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' '
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=ste
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=ste
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=el
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=el
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [debug    ] gemini_input_transcript        text=' recording.'
2026-03-24 07:02:15 [info     ] user_speech_transcribed        session_id=v5_mph3j0z2 text=' recording.'
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:15 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:15 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:16 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] mic_chunk_received             bytes=8192 session_id=v5_mph3j0z2
2026-03-24 07:02:16 [debug    ] gemini_sending_audio_chunk     bytes=8192
2026-03-24 07:02:32 [info     ] browser_message_received       message_type=websocket.receive session_id=v5_mph3j0z2
2026-03-24 07:02:32 [debug    ] text_message_parsed_as_json    text='This is a problem'
2026-03-24 07:02:32 [info     ] text_message_received          preview='This is a problem' session_id=v5_mph3j0z2
2026-03-24 07:02:32 [debug    ] gemini_sending_text            text='This is a problem'